**UTKFace — A1–A12, 18 cấu hình × 3 seed = 54 lượt train.**

Chạy lần đầu: để `RESUME_FROM=''`, chọn GPU, rồi **Save Version → Run all and save**. Notebook tự tạm dừng theo `MAX_SESSION_HOURS` để lưu output. Chạy tiếp: **Add Input → Notebook Output** phiên trước + dataset UTKFace, đặt `RESUME_FROM` tới thư mục chứa `config.json`, `data/`, `runs/`, rồi Run all. Giữ nguyên code và thông số nghiên cứu. Không dùng output bộ E cũ để resume bộ A mới.

`all_results.json` chứa toàn bộ kết quả, bảng XLSX, logs, splits và số liệu tài nguyên; checkpoint `.pt` giữ riêng trong cùng output để resume. Cần output đã được Kaggle lưu; hard kill trước khi lưu output không thể bảo đảm phục hồi. Chuyển GPU/phiên bản thư viện có thể gây sai khác số học. Chi phí tiền chỉ tính nếu nhập giá GPU-hour.

In [1]:
import os, sys, time
from pathlib import Path
os.environ['OMP_NUM_THREADS']='2'
os.environ['MKL_NUM_THREADS']='2'
os.environ['OPENBLAS_NUM_THREADS']='1'
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF','max_split_size_mb:128')

DATA_ROOT = '/kaggle/input/datasets/jangedoo/utkface-new/UTKFace'
OUT = '/kaggle/working/utkface_A12_3seeds_v1'
CACHE_ROOT = '/kaggle/temp/utkface_A12_cache'  # Rebuilt from original input when moving machines.
RESUME_FROM = ''  # Full old OUTPUT folder containing config.json, data/, runs/.
# Example: '/kaggle/input/<your-notebook-output>/utkface_A12_3seeds_v1'
WEIGHTS_PATH = ''  # Resume reuses pretrained weights saved under OUT/assets automatically.
TRAINING_SEEDS = [42, 2006, 2026]
MAX_SESSION_HOURS = 10.0  # Set LOWER than your remaining session/GPU quota; user-controlled budget.
GPU_HOUR_PRICE_USD = None  # Optional actual price per GPU-hour; None means no invented monetary cost.
CODE_DIR = Path('/kaggle/working/utkface_A12_code_v1')
CODE_DIR.mkdir(parents=True,exist_ok=True)
sys.path.insert(0,str(CODE_DIR)) if str(CODE_DIR) not in sys.path else None
CFG = dict(
    schema_version='utkface_A12_multiseed_v1', data_root=DATA_ROOT, out=OUT, cache_root=CACHE_ROOT,
    resume_from=RESUME_FROM, weights_path=WEIGHTS_PATH, training_seeds=TRAINING_SEEDS,
    split_seed=42, fold_index=None, image_size=224, dropout=0.2, freeze_bn_stats=True,
    deterministic=True, batch_size=32, eval_batch_size=64, learning_rate=3e-4,
    weight_decay=1e-4, beta=0.999, gamma=2.0, epochs=50, patience=10, gradient_clip=5.0,
    checkpoint_every_batches=50, checkpoint_seconds=300, keep_completed_optimizer=False,
    timing_repeats=20, bootstrap_replicates=1000, json_snapshot_seconds=180,
    log_poll_seconds=2, resource_sample_seconds=10, max_session_hours=MAX_SESSION_HOURS,
    shutdown_margin_seconds=600, gpu_hour_price_usd=GPU_HOUR_PRICE_USD,
    export_xlsx=True, evaluate_train_at_best=True, gpu_ids=[], smoke_testing=False)
# Same split and training setup for all methods. Do NOT change source/hyperparameters when resuming.
# Keep completed best.pt; last.pt includes model/optimizer/scheduler/scaler/RNG/cursor for unfinished runs.


In [2]:
import importlib.util, subprocess
needed={'numpy':'numpy','pandas':'pandas','PIL':'Pillow','sklearn':'scikit-learn',
        'matplotlib':'matplotlib','tqdm':'tqdm','psutil':'psutil','openpyxl':'openpyxl'}
missing=[pkg for mod,pkg in needed.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install','--quiet',*missing])
import torch, torchvision, psutil
if not torch.cuda.is_available(): raise RuntimeError('Bật GPU trong Kaggle Settings trước khi Run all.')
print('GPUs:',[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
print('CPU logical:',psutil.cpu_count(),'RAM GiB:',round(psutil.virtual_memory().total/1024**3,1))


GPUs: ['Tesla T4', 'Tesla T4']
CPU logical: 4 RAM GiB: 31.3


In [3]:
%%writefile /kaggle/working/utkface_A12_code_v1/data.py

"""Chuẩn bị UTKFace một lần: kiểm tra ảnh, bỏ trùng, chia tập, cache uint8 trên đĩa."""

import os, json, hashlib, shutil, math
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
from sklearn.model_selection import train_test_split, StratifiedKFold
from tqdm.auto import tqdm

LABELS = ["0-9", "10-19", "20-29", "30-39", "40-49", "50-59", "60-69", "70+"]


def atomic_json(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
    os.replace(tmp, path)


def fingerprint(obj):
    return hashlib.sha256(json.dumps(obj, sort_keys=True, ensure_ascii=False).encode()).hexdigest()


def inspect_image(args):
    file, root = args
    try:
        token = file.name.split("_")[0]
        if not token.isdigit():
            raise ValueError("Tên không có tuổi nguyên hợp lệ")
        age = int(token)
        if not 0 <= age <= 116:
            raise ValueError("Tuổi ngoài 0..116")
        with Image.open(file) as im:
            im = ImageOps.exif_transpose(im).convert("RGB")
            arr = np.asarray(im)
            sha = hashlib.sha256(str(arr.shape).encode() + arr.tobytes()).hexdigest()
        return dict(
            file=str(file.relative_to(root)),
            age=age,
            label=min(age // 10, 7),
            sha256=sha,
            width=arr.shape[1],
            height=arr.shape[0],
            error="",
        )
    except Exception as e:
        return dict(file=str(file.relative_to(root)), error=f"{type(e).__name__}: {e}")


def prepare_data(cfg):
    root = Path(cfg["data_root"])
    out = Path(cfg["out"]) / "data"
    out.mkdir(parents=True, exist_ok=True)
    if not root.is_dir():
        raise FileNotFoundError(f"Không thấy UTKFace tại {root}. Hãy Add Input đúng dataset.")
    files = sorted(
        p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in (".jpg", ".jpeg", ".png")
    )
    if not files:
        raise RuntimeError("Không có ảnh trong thư mục dữ liệu.")
    # Kiểm tra inventory mỗi lần; full pixel hash lần đầu hoặc khi danh sách/kích thước file đổi.
    inventory = fingerprint(
        [(str(p.relative_to(root)), p.stat().st_size, p.stat().st_mtime_ns) for p in files]
    )
    invfile = out / "inventory.json"
    if (
        invfile.exists()
        and json.loads(invfile.read_text()).get("fingerprint") == inventory
        and (out / "audit.csv").exists()
    ):
        audit = pd.read_csv(out / "audit.csv", keep_default_na=False)
    else:
        workers = max(1, min(4, os.cpu_count() or 1))
        with ThreadPoolExecutor(max_workers=workers) as pool:
            audit = pd.DataFrame(
                tqdm(
                    pool.map(inspect_image, ((f, root) for f in files)),
                    total=len(files),
                    desc="Kiểm tra và hash ảnh",
                    mininterval=2,
                    miniters=max(1, len(files) // 100),
                )
            )
        audit.to_csv(out / "audit.csv", index=False)
        atomic_json({"fingerprint": inventory}, invfile)
    bad = audit[audit.error != ""]
    bad.to_csv(out / "rejected.csv", index=False)
    valid = audit[audit.error == ""].copy()
    # audit.csv có ô trống của ảnh hỏng: pandas có thể lưu tuổi thành "12.0".
    # Đọc lại phải chuyển chuỗi số sang numeric trước, nếu không resume sẽ lỗi.
    valid[["age", "label"]] = (
        valid[["age", "label"]].apply(pd.to_numeric, errors="raise").astype(np.int64)
    )
    inconsistent = valid.groupby("sha256").age.nunique()
    conflict_ids = inconsistent[inconsistent > 1].index
    valid[valid.sha256.isin(conflict_ids)].to_csv(out / "conflicting_duplicates.csv", index=False)
    clean = valid[~valid.sha256.isin(conflict_ids)].copy()
    duplicates = clean[clean.duplicated("sha256", keep=False)]
    duplicates.to_csv(out / "duplicates.csv", index=False)
    clean = clean.drop_duplicates("sha256", keep="first").sort_values("file").reset_index(drop=True)
    clean["cache_index"] = np.arange(len(clean))
    counts = clean.label.value_counts().reindex(range(8), fill_value=0)
    if (counts < 20).any():
        raise ValueError(f"Có lớp <20 ảnh; cần kiểm tra dữ liệu trước khi chia: {counts.to_dict()}")
    indices = np.arange(len(clean))
    labels = clean.label.to_numpy()
    seed = cfg["split_seed"]
    fold = cfg.get("fold_index")
    if fold is None:
        trainval, test = train_test_split(
            indices, test_size=0.1, random_state=seed, stratify=labels
        )
        train, val = train_test_split(
            trainval, test_size=1 / 9, random_state=seed + 1, stratify=labels[trainval]
        )
    else:
        if fold not in range(5):
            raise ValueError("fold_index phải là None hoặc 0..4.")
        dev, test = list(
            StratifiedKFold(5, shuffle=True, random_state=seed).split(indices, labels)
        )[fold]
        train, val = train_test_split(
            dev, test_size=0.125, random_state=seed + 1, stratify=labels[dev]
        )
    clean["split"] = ""
    for name, ids in [("train", train), ("val", val), ("test", test)]:
        clean.loc[ids, "split"] = name
    for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
        assert not set(clean.loc[clean.split == a, "sha256"]) & set(
            clean.loc[clean.split == b, "sha256"]
        )
    content_fp = fingerprint(clean[["file", "age", "label", "sha256"]].to_dict("records"))
    split_fp = fingerprint(clean[["sha256", "label", "split"]].to_dict("records"))
    old_report=out/'split_report.json'
    if old_report.exists() and json.loads(old_report.read_text())['split_fingerprint']!=split_fp:
        raise ValueError('Dataset hoặc split khác output cũ. Không ghi đè split; dùng OUT mới.')
    for name in ["train", "val", "test"]:
        clean[clean.split == name].to_csv(out / f"{name}.csv", index=False)
    clean.to_csv(out / "manifest.csv", index=False)
    # Cache nằm ngoài thư mục kết quả: không đóng gói lại dataset ảnh vào ZIP kết quả.
    cache_dir = Path(cfg["cache_root"])
    cache_dir.mkdir(parents=True, exist_ok=True)
    cache_key = fingerprint([content_fp, cfg["image_size"], "RGB_bilinear_v1"])[:20]
    cache = cache_dir / f"images_{cache_key}.npy"
    meta = cache.with_suffix(".json")
    shape = (len(clean), cfg["image_size"], cfg["image_size"], 3)
    required = int(np.prod(shape)) + 4096
    rebuild = True
    if cache.exists() and meta.exists():
        try:
            mm = np.load(cache, mmap_mode="r")
            rebuild = mm.shape != shape or mm.dtype != np.uint8
            del mm
        except Exception:
            rebuild = True
    if rebuild:
        if shutil.disk_usage(cache_dir).free < required + 2 * 1024**3:
            raise OSError("Không đủ đĩa để tạo cache và chừa 2 GiB cho checkpoint.")
        tmp = cache.with_name(cache.stem + ".partial.npy")
        mm = np.lib.format.open_memmap(tmp, mode="w+", dtype=np.uint8, shape=shape)

        def load(row):
            with Image.open(root / row.file) as im:
                im = (
                    ImageOps.exif_transpose(im)
                    .convert("RGB")
                    .resize((cfg["image_size"],) * 2, Image.Resampling.BILINEAR)
                )
                return np.asarray(im, dtype=np.uint8)

        with ThreadPoolExecutor(max_workers=max(1, min(4, os.cpu_count() or 1))) as pool:
            rows = list(clean.itertuples(index=False))

            def bounded_images():
                for start in range(0, len(rows), 128):
                    yield from pool.map(load, rows[start : start + 128])

            iterator = bounded_images()
            for i, arr in enumerate(
                tqdm(
                    iterator,
                    total=len(clean),
                    desc="Cache ảnh uint8",
                    mininterval=2,
                    miniters=max(1, len(clean) // 100),
                )
            ):
                mm[i] = arr
                if i % 512 == 0:
                    mm.flush()
        mm.flush()
        del mm
        os.replace(tmp, cache)
        atomic_json(
            {"content_fingerprint": content_fp, "shape": list(shape), "dtype": "uint8"}, meta
        )
    report = {
        "inventory_fingerprint": inventory,
        "content_fingerprint": content_fp,
        "split_fingerprint": split_fp,
        "source_files": len(files),
        "valid_files": len(valid),
        "rejected": len(bad),
        "conflicting_files": int(valid.sha256.isin(conflict_ids).sum()),
        "duplicate_groups": int(duplicates.sha256.nunique()),
        "redundant_removed": int(len(valid) - valid.sha256.isin(conflict_ids).sum() - len(clean)),
        "retained": len(clean),
        "cache_path": str(cache),
        "cache_gib": required / 1024**3,
        "split_seed": seed,
        "fold_index": fold,
        "class_labels": LABELS,
        "split_counts": pd.crosstab(clean.split, clean.label)
        .reindex(columns=range(8), fill_value=0)
        .to_dict("index"),
        "identity_limitation": "Pixel-disjoint, không phải person-disjoint; UTKFace không cung cấp subject ID đáng tin cậy trong tên file.",
    }
    atomic_json(report, out / "split_report.json")
    return report


def epoch_batches(frame, batch_size, seed, balanced=False):
    """Danh sách cache_index, thứ tự tái lập độc lập với trạng thái model/loss."""
    rng = np.random.default_rng(seed)
    ids = frame.cache_index.to_numpy()
    if balanced:
        raise ValueError("Bản E-only sử dụng cùng sampler ngẫu nhiên cho mọi phương pháp.")
    ids = rng.permutation(ids)
    return [ids[i : i + batch_size] for i in range(0, len(ids), batch_size)]

Writing /kaggle/working/utkface_A12_code_v1/data.py


In [4]:
%%writefile /kaggle/working/utkface_A12_code_v1/specs.py

"""Single source of truth: 12 ablations, 18 configurations, 3 training seeds."""

def experiment_specs():
    rows = [
        ('A1','C','Chỉ C có kém phương pháp đầy đủ không?'),
        ('A2','O','Phương pháp đầy đủ có tốt hơn ordinal thuần không?'),
        ('A3','sg(w_y D) O','Thành phần C đóng góp gì khi giữ nguyên nhánh ordinal?'),
    ]
    specs = [dict(id=m, family=m, formula=f, question=q) for m,f,q in rows]
    for i, value in enumerate([0.03, 0.05, 0.1, 0.3, 0.5, 1.0, 3.0],1):
        specs.append(dict(id=f'A4_{i}', family='A4', lam=value, formula=f'C + {value:g} O',
            question='Lambda thích nghi có tốt hơn lambda cố định được chọn bằng validation không?'))
    rows = [
        ('A5','C + sg(w_y n_eff) O','Dùng trực tiếp n_eff có tốt hơn ánh xạ D=(1+s)/2 không?'),
        ('A6','C + sg(D) O','Ordinal có cần trọng số lớp khi C đã được cân theo lớp không?'),
        ('A7','C + w_y O','Điều chỉnh theo cấu trúc sai lệch có tốt hơn chỉ dùng trọng số lớp không?'),
        ('A8','-w_y log(p_y) + sg(w_y D) O','Focal trong C có cần thiết khi đã có ordinal thích nghi không?'),
        ('A9','C + w_y D O (gradient through lambda)','Cho gradient đi qua lambda có tốt hơn stop-gradient không?'),
        ('A10','-(1-p_y)^gamma log(p_y) + sg(w_y D) O','C có cần trọng số lớp khi ordinal đã được cân theo lớp không?'),
        ('A11','C + sg(w_y (1-p_y)^gamma D) O','Thêm focal vào ordinal có tốt hơn hay không?'),
        ('A12','C + sg(w_y D) O','Phương pháp đầy đủ; mốc so sánh các ablation.'),
    ]
    return specs + [dict(id=m,family=m,formula=f,question=q) for m,f,q in rows]


def build_plan(cfg):
    seeds = cfg['training_seeds']
    if len(seeds)!=len(set(seeds)) or not seeds or any(type(s) is not int or not 0<=s<2**32 for s in seeds):
        raise ValueError('training_seeds phải là danh sách seed nguyên, không trùng.')
    specs=experiment_specs()
    return dict(schema_version='utkface_A12_multiseed_v1', training_seeds=seeds,
        number_of_training_seeds=len(seeds), configurations_per_seed=len(specs),
        total_train_runs=len(specs)*len(seeds),total_test_evaluations=len(specs)*len(seeds),
        primary_method='A12',primary_metric='qwk',specs=specs,
        fixed_lambda_selection='One lambda, ranked by mean validation QWK across all training seeds; tie: mean MAE then macro-F1.',
        split_note='Same split_seed for all methods and training seeds; seeds vary training only.')


Writing /kaggle/working/utkface_A12_code_v1/specs.py


In [5]:
%%writefile /kaggle/working/utkface_A12_code_v1/losses.py

"""Các loss và head ordinal. Ký hiệu thống nhất với kịch bản thực nghiệm."""

import math
import torch
from torch import nn
import torch.nn.functional as F


def cb_weights(counts, beta=0.999):
    """Căn bậc hai của inverse effective number, sau đó chuẩn hóa mean=1."""
    n = torch.as_tensor(counts, dtype=torch.float64)
    if (n <= 0).any():
        raise ValueError("Tập train phải có đủ mọi lớp.")
    if not 0 <= beta < 1:
        raise ValueError("beta phải thuộc [0,1).")
    a = torch.ones_like(n) if beta == 0 else (1 - beta) / (-torch.expm1(n * math.log(beta)))
    w = a.sqrt()
    return (w / w.mean()).float()


def ordinal_parts(logits, y, weights, gamma):
    """Log-sum-exp tránh log(0), không clamp q làm mất gradient ở mẫu khó."""
    z = logits if logits.dtype == torch.float64 else logits.float()
    K = z.shape[1]
    logp = F.log_softmax(z, dim=1)
    p = logp.exp()
    t = y[:, None] > torch.arange(K - 1, device=z.device)
    log_le = torch.logcumsumexp(logp, dim=1)[:, :-1]
    log_gt = torch.logcumsumexp(logp.flip(1), dim=1).flip(1)[:, 1:]
    ordinal = -torch.where(t, log_gt, log_le).mean(1)
    # Tính trực tiếp tổng xác suất phía sai; tránh 1-q gây triệt tiêu số học.
    log_e = torch.where(t, log_le, log_gt)
    e = log_e.exp()
    # Chuẩn hóa theo max trước khi bình phương: cùng tỷ số nhưng không underflow.
    scaled = torch.exp(log_e - log_e.max(1, keepdim=True).values)
    neff = scaled.sum(1).square() / scaled.square().sum(1)
    neff = neff.clamp(1.0, float(K - 1))
    s = (neff - 1) / (K - 2)
    true_logp = logp.gather(1, y[:, None]).squeeze(1)
    # log(1-p_y) bằng log-sum-exp các lớp khác, ổn định cả khi p_y~1.
    other = logp.masked_fill(F.one_hot(y, K).bool(), -torch.inf)
    delta = torch.logsumexp(other, dim=1).exp()
    focal = delta.pow(gamma)
    wf = weights[y].to(z.dtype) * focal
    cls = -wf * true_logp
    return dict(
        p=p, C=cls, O=ordinal, F=wf, focal=focal, neff=neff, s=s, error_sum=e.sum(1), sbar=e.mean(1)
    )


class LossBank(nn.Module):
    def __init__(self, spec, counts, beta=0.999, gamma=2.0):
        super().__init__()
        self.spec, self.K, self.gamma = spec, len(counts), gamma
        self.register_buffer("weights", cb_weights(counts, beta))

    def terms(self, z, y):
        a = ordinal_parts(z, y, self.weights, self.gamma)
        logpy = F.log_softmax(z, dim=1).gather(1, y[:, None]).squeeze(1)
        wy = self.weights[y].to(z.dtype)
        D = (1 + a['s']) / 2
        family = self.spec['family']
        cls = a['C']
        lam = wy * D
        if family == 'A1': lam = torch.zeros_like(cls)
        elif family == 'A2': cls = torch.zeros_like(cls); lam = torch.ones_like(lam)
        elif family == 'A3': cls = torch.zeros_like(cls)
        elif family == 'A4': lam = torch.full_like(lam, self.spec['lam'])
        elif family == 'A5': lam = wy * a['neff']
        elif family == 'A6': lam = D
        elif family == 'A7': lam = wy
        elif family == 'A8': cls = -wy * logpy
        elif family == 'A9': pass
        elif family == 'A10': cls = -a['focal'] * logpy
        elif family == 'A11': lam = wy * a['focal'] * D
        elif family != 'A12': raise ValueError(f'Unknown ablation: {family}')
        if family != 'A9': lam = lam.detach()
        weighted_O = lam * a['O']
        return cls, weighted_O, lam, a

    def forward(self, out, y, generator=None):
        z = out['logits'] if isinstance(out, dict) else out
        z = z if z.dtype == torch.float64 else z.float()
        cls, weighted_O, lam, a = self.terms(z, y)
        diagnostics = {k:a[k].detach() for k in ('O','neff','s','error_sum')}
        # C is the ACTUAL classification term of this ablation; common_C retained separately.
        diagnostics.update(C=cls.detach(),common_C=a['C'].detach(),lam=lam.detach(),weighted_O=weighted_O.detach())
        return (cls + weighted_O).mean(), diagnostics


def predict(out, family):
    p = out["logits"].float().softmax(1)
    return p.argmax(1), p, 0.0


def sampled_logit_gradients(logits, y, criterion):
    """Actual logit gradients of each loss term, including O*grad(lambda) for A9.
    Zero classification/ordinal gradient yields undefined cosine/ratio -> JSON null.
    """
    z = logits.detach().double().requires_grad_(True)
    cls, ordinal, _, _ = criterion.terms(z,y)
    def grad(term, retain):
        if not term.requires_grad: return torch.zeros_like(z)
        return torch.autograd.grad(term.sum(),z,retain_graph=retain)[0]
    gc = grad(cls,True); go = grad(ordinal,False)
    nc,no = gc.abs().sum(1),go.abs().sum(1)
    tiny=torch.finfo(z.dtype).tiny
    ratio=torch.where(nc>0,no/nc.clamp_min(tiny),torch.full_like(nc,float('nan')))
    sc,so=gc.abs().amax(1),go.abs().amax(1)
    uc,uo=gc/sc.clamp_min(tiny)[:,None],go/so.clamp_min(tiny)[:,None]
    cosine=(uc*uo).sum(1)/(uc.norm(dim=1)*uo.norm(dim=1)).clamp_min(tiny)
    cosine=torch.where((sc>0)&(so>0),cosine.clamp(-1,1),torch.full_like(sc,float('nan')))
    return torch.stack([nc,no,ratio,cosine],1).detach()


Writing /kaggle/working/utkface_A12_code_v1/losses.py


In [6]:
%%writefile /kaggle/working/utkface_A12_code_v1/models.py

"""Cùng ResNeXt50 và head softmax 8 lớp cho mọi kịch bản E."""

import torch
from torch import nn
from torch.utils.checkpoint import checkpoint
from torchvision.models import resnext50_32x4d
import torch.nn.functional as F


class OrdinalModel(nn.Module):
    def __init__(self, cfg, family):
        super().__init__()
        self.family = family
        self.K = 8
        if cfg.get("smoke_testing", False):
            self.encoder = nn.Sequential(
                nn.Conv2d(3, 8, 3, 2, 1), nn.ReLU(), nn.AdaptiveAvgPool2d(1), nn.Flatten()
            )
            dim = 8
        else:
            self.encoder = resnext50_32x4d(weights=None)
            state = torch.load(cfg["weights_path"], map_location="cpu", weights_only=True)
            self.encoder.load_state_dict(state, strict=True)
            dim = self.encoder.fc.in_features
            self.encoder.fc = nn.Identity()
        self.dropout = nn.Dropout(cfg["dropout"])
        self.head = nn.Linear(dim, 8)
        self.freeze_bn = cfg["freeze_bn_stats"]

    def train(self, mode=True):
        super().train(mode)
        if mode and self.freeze_bn:
            for module in self.encoder.modules():
                if isinstance(module, nn.modules.batchnorm._BatchNorm):
                    module.eval()
        return self

    def forward(self, x, microbatch=None):
        microbatch = microbatch or len(x)
        if self.training and microbatch < len(x):
            # Giữ nguyên logical batch cho hoán đổi lambda.
            # Checkpoint tính lại encoder khi backward để giảm activation memory.
            feat = torch.cat(
                [checkpoint(self.encoder, c, use_reentrant=False) for c in x.split(microbatch)], 0
            )
        else:
            feat = self.encoder(x)
        feat = self.dropout(feat)
        return {"logits": self.head(feat)}


def preprocess(batch, device, training, seed):
    """Chuyển uint8 lên GPU rồi augmentation. CPU không resize/augment trong train."""
    x = torch.from_numpy(batch).to(device, non_blocking=False).permute(0, 3, 1, 2).float() / 255.0
    if training:
        gen = torch.Generator(device=device).manual_seed(seed)
        B = len(x)
        rand = lambda shape: torch.rand(shape, device=device, generator=gen)
        flip = rand((B, 1, 1, 1)) < 0.5
        x = torch.where(flip, x.flip(3), x)
        angle = (rand((B,)) - 0.5) * (20 * torch.pi / 180)
        scale = 0.90 + 0.10 * rand((B,))
        theta = torch.zeros((B, 2, 3), device=device)
        theta[:, 0, 0] = scale * angle.cos()
        theta[:, 0, 1] = -scale * angle.sin()
        theta[:, 1, 0] = scale * angle.sin()
        theta[:, 1, 1] = scale * angle.cos()
        theta[:, :, 2] = (rand((B, 2)) - 0.5) * 0.08
        grid = F.affine_grid(theta, x.shape, align_corners=False)
        x = F.grid_sample(x, grid, mode="bilinear", padding_mode="reflection", align_corners=False)
        brightness = 0.9 + 0.2 * rand((B, 1, 1, 1))
        contrast = 0.9 + 0.2 * rand((B, 1, 1, 1))
        mean = x.mean((2, 3), keepdim=True)
        x = ((x - mean) * contrast + mean) * brightness
        x = x.clamp(0, 1)
    mean = x.new_tensor([0.485, 0.456, 0.406])[None, :, None, None]
    std = x.new_tensor([0.229, 0.224, 0.225])[None, :, None, None]
    return ((x - mean) / std).contiguous()

Writing /kaggle/working/utkface_A12_code_v1/models.py


In [7]:
%%writefile /kaggle/working/utkface_A12_code_v1/metrics.py

"""Metric theo 8 chỉ số nhóm tuổi, KHÔNG phải sai số tuổi tính bằng năm."""

import numpy as np
from sklearn.metrics import (
    accuracy_score,
    cohen_kappa_score,
    precision_recall_fscore_support,
    confusion_matrix,
    matthews_corrcoef,
    roc_auc_score,
    average_precision_score,
)


def finite_or_none(x):
    return float(x) if np.isfinite(x) else None


def compute_metrics(y, pred, probs=None):
    y = np.asarray(y, dtype=int)
    pred = np.asarray(pred, dtype=int)
    K = 8
    cm = confusion_matrix(y, pred, labels=np.arange(K))
    n = cm.sum()
    pr, re, f1, support = precision_recall_fscore_support(
        y, pred, labels=np.arange(K), zero_division=0
    )
    tp = np.diag(cm)
    fp = cm.sum(0) - tp
    fn = cm.sum(1) - tp
    tn = n - tp - fp - fn
    present = support > 0
    result = {
        "n": int(len(y)),
        "accuracy": float(accuracy_score(y, pred)),
        "qwk": finite_or_none(cohen_kappa_score(y, pred, labels=np.arange(K), weights="quadratic")),
        "group_mae": float(np.abs(y - pred).mean()),
        "balanced_accuracy": float(re[present].mean()),
        "precision_macro": float(pr.mean()),
        "recall_macro": float(re.mean()),
        "f1_macro": float(f1.mean()),
        "f1_weighted": float(np.average(f1, weights=support)),
        "precision_weighted": float(np.average(pr, weights=support)),
        "recall_weighted": float(np.average(re, weights=support)),
        "within1": float((np.abs(y - pred) <= 1).mean()),
        "severe_error_rate": float((np.abs(y - pred) >= 2).mean()),
        "specificity_macro": float(
            np.divide(tn, tn + fp, out=np.zeros(K, dtype=float), where=(tn + fp) > 0).mean()
        ),
        "mcc": finite_or_none(matthews_corrcoef(y, pred)),
        "confusion_matrix": cm.tolist(),
        "per_class": [
            {
                "class": i,
                "support": int(support[i]),
                "precision": float(pr[i]),
                "recall": float(re[i]),
                "f1": float(f1[i]),
                "mae": float(np.abs(pred[y == i] - i).mean()) if support[i] else None,
            }
            for i in range(K)
        ],
        "nll": None,
        "ece15": None,
        "auc_roc_macro": None,
        "auc_pr_macro": None,
    }
    if probs is not None:
        probs = np.asarray(probs, dtype=np.float64)
        if not np.isfinite(probs).all() or (probs < -1e-6).any():
            raise ValueError("Xác suất không hợp lệ.")
        probs = np.maximum(probs, 0)
        probs /= probs.sum(1, keepdims=True)
        result["nll"] = float(-np.log(np.maximum(probs[np.arange(len(y)), y], 1e-15)).mean())
        mode = probs.argmax(1)
        conf = probs.max(1)
        correct = mode == y
        bins = np.minimum((conf * 15).astype(int), 14)
        ece = 0.0
        for b in range(15):
            m = bins == b
            if m.any():
                ece += m.mean() * abs(correct[m].mean() - conf[m].mean())
        result["ece15"] = float(ece)
        result["qwk_argmax"] = finite_or_none(
            cohen_kappa_score(y, mode, labels=np.arange(K), weights="quadratic")
        )
        result["accuracy_argmax"] = float((mode == y).mean())
        mean_pred = np.floor((probs * np.arange(K)).sum(1) + 0.5).clip(0, K - 1).astype(int)
        result["qwk_rounded_expectation"] = finite_or_none(
            cohen_kappa_score(y, mean_pred, labels=np.arange(K), weights="quadratic")
        )
        result["group_mae_rounded_expectation"] = float(np.abs(y - mean_pred).mean())
        onehot = np.eye(K)[y]
        if present.all():
            result["auc_roc_macro"] = float(roc_auc_score(onehot, probs, average="macro"))
            result["auc_pr_macro"] = float(average_precision_score(onehot, probs, average="macro"))
    distance=np.abs(y-pred)
    result.update(group_rmse=float(np.sqrt(np.mean((y-pred)**2))),
        linear_weighted_kappa=finite_or_none(cohen_kappa_score(y,pred,labels=np.arange(K),weights='linear')),
        within2=float((distance<=2).mean()),mean_signed_group_error=float((pred-y).mean()),
        underestimation_rate=float((pred<y).mean()),overestimation_rate=float((pred>y).mean()),
        error_distance_counts=np.bincount(distance,minlength=K).tolist(),
        confusion_matrix_normalized_true=np.divide(cm,support[:,None],out=np.zeros_like(cm,dtype=float),where=support[:,None]>0).tolist(),
        confusion_matrix_normalized_pred=np.divide(cm,cm.sum(0)[None,:],out=np.zeros_like(cm,dtype=float),where=cm.sum(0)[None,:]>0).tolist())
    for i,row in enumerate(result['per_class']):
        row.update(tp=int(tp[i]),fp=int(fp[i]),fn=int(fn[i]),tn=int(tn[i]),
            specificity=float(tn[i]/(tn[i]+fp[i])) if tn[i]+fp[i] else None,
            auc_roc=None,auc_pr=None)
        if probs is not None and 0<support[i]<n:
            row['auc_roc']=float(roc_auc_score((y==i).astype(int),probs[:,i]))
            row['auc_pr']=float(average_precision_score((y==i).astype(int),probs[:,i]))
    if probs is not None:
        result['brier_multiclass']=float(np.square(probs-np.eye(K)[y]).sum(1).mean())
        result['calibration_bins_15']=[dict(bin=b,lower=b/15,upper=(b+1)/15,
            n=int((bins==b).sum()),mean_confidence=float(conf[bins==b].mean()) if (bins==b).any() else None,
            accuracy=float(correct[bins==b].mean()) if (bins==b).any() else None) for b in range(15)]
    return result


def checkpoint_key(m):
    # QWK lớn hơn; hòa tới 1e-8 thì MAE nhỏ hơn, rồi Macro-F1 lớn hơn.
    return (
        round(m["qwk"] if m["qwk"] is not None else -2.0, 8),
        -round(m["group_mae"], 8),
        round(m["f1_macro"], 8),
    )

Writing /kaggle/working/utkface_A12_code_v1/metrics.py


In [8]:
%%writefile /kaggle/working/utkface_A12_code_v1/resources.py

"""Per-attempt resource ledger; actual time includes failed/replayed work. No fabricated prices/FLOPs."""
import os, time, json, platform, subprocess, threading, uuid
from pathlib import Path
import psutil
import resource
from data import atomic_json

def utc():
    return time.strftime('%Y-%m-%dT%H:%M:%SZ',time.gmtime())

def hardware():
    import torch, torchvision
    return dict(python=platform.python_version(),torch=torch.__version__,torchvision=torchvision.__version__,
        platform=platform.platform(),cpu_model=platform.processor(),cpu_logical=psutil.cpu_count(),
        cpu_physical=psutil.cpu_count(logical=False),ram_total_gib=psutil.virtual_memory().total/1024**3,
        cuda=torch.version.cuda,cudnn=torch.backends.cudnn.version(),
        gpu_names=[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

class ResourceMonitor:
    def __init__(self,folder,cfg,mode,job=None):
        self.folder=Path(folder); self.folder.mkdir(parents=True,exist_ok=True)
        self.cfg,self.mode,self.job=cfg,mode,job or {}
        self.id=f'{time.time_ns()}_{os.getpid()}_{uuid.uuid4().hex[:6]}'
        self.base=self.folder/f'{mode}_{self.id}'
        self.stop_event=threading.Event()
        self.start=time.monotonic(); self.started=utc(); self.cpu0=time.process_time(); self.cpu_previous=(self.start,self.cpu0)
        self.peak=0;self.samples=0;self.util=[];self.power=[];self.energy=0.;self.power_seconds=0.;self.previous=None
        self.gpu_index=self.job.get('physical_gpu_id')
        self.gpu_count=int(mode in ('train','test') and self.gpu_index is not None)
        self.meta=dict(attempt_id=self.id,mode=mode,run_id=self.job.get('run_id'),seed=self.job.get('seed'),
            started_at_utc=self.started,hardware=hardware(),physical_gpu_id=self.gpu_index)
        psutil.cpu_percent(None)
        self.sample()
        self.thread=threading.Thread(target=self.loop,daemon=True);self.thread.start()
    def gpu(self):
        if self.gpu_index is None:return None
        try:
            result=subprocess.run(['nvidia-smi',f'--id={self.gpu_index}',
                '--query-gpu=uuid,name,utilization.gpu,memory.used,memory.total,power.draw,temperature.gpu',
                '--format=csv,noheader,nounits'],capture_output=True,text=True,timeout=2,check=True)
            v=[x.strip() for x in result.stdout.strip().split(',')]
            def num(x):
                try:return float(x)
                except ValueError:return None
            return dict(uuid=v[0],name=v[1],utilization_percent=num(v[2]),memory_used_mib=num(v[3]),
                memory_total_mib=num(v[4]),power_watts=num(v[5]),temperature_c=num(v[6]))
        except Exception as e:return {'unavailable':str(e)}
    def sample(self):
        now=time.monotonic();cpu=time.process_time()
        usage=resource.getrusage(resource.RUSAGE_SELF)
        rss=usage.ru_maxrss*1024
        try:
            for line in Path('/proc/self/status').read_text().splitlines():
                if line.startswith('VmRSS:'):rss=int(line.split()[1])*1024;break
        except OSError:pass
        self.peak=max(self.peak,rss,usage.ru_maxrss*1024); self.samples+=1
        cpu_percent=100*(cpu-self.cpu_previous[1])/max(now-self.cpu_previous[0],1e-9)
        self.cpu_previous=(now,cpu);gpu=self.gpu()
        row=dict(at_utc=utc(),elapsed_seconds=now-self.start,process_cpu_percent=cpu_percent,
            host_cpu_percent=psutil.cpu_percent(None),process_rss_gib=rss/1024**3,
            host_ram_used_gib=psutil.virtual_memory().used/1024**3,
            host_ram_available_gib=psutil.virtual_memory().available/1024**3,
            disk_free_gib=psutil.disk_usage(self.folder).free/1024**3,
            process_cpu_seconds=cpu-self.cpu0,gpu=gpu)
        if gpu and gpu.get('utilization_percent') is not None:self.util.append(gpu['utilization_percent'])
        if gpu and gpu.get('power_watts') is not None:
            power=gpu['power_watts']; self.power.append(power)
            if self.previous:
                dt=now-self.previous[0]
                self.energy+=(power+self.previous[1])/2*dt/3600000;self.power_seconds+=dt
            self.previous=(now,power)
        else:self.previous=None
        with self.base.with_suffix('.jsonl').open('a',encoding='utf-8') as f:
            f.write(json.dumps(row,ensure_ascii=False,allow_nan=False)+'\n');f.flush()
        self.write_summary('running')
    def write_summary(self,status,error=None):
        elapsed=time.monotonic()-self.start;cpu=time.process_time()
        gpu_hours=elapsed*self.gpu_count/3600
        rate=self.cfg.get('gpu_hour_price_usd')
        atomic_json({**self.meta,'status':status,'updated_at_utc':utc(),'error':error,
            'wall_seconds':elapsed,'process_cpu_seconds':cpu-self.cpu0,
            'peak_process_rss_gib':self.peak/1024**3,'samples':self.samples,
            'assigned_gpu_hours':gpu_hours,'gpu_hour_price_usd':rate,
            'estimated_cost_usd':gpu_hours*rate if rate is not None and self.gpu_count else None,
            'gpu_utilization_sample_mean':sum(self.util)/len(self.util) if self.util else None,
            'gpu_power_watts_sample_mean':sum(self.power)/len(self.power) if self.power else None,
            'sampled_gpu_energy_kwh':self.energy if self.power_seconds else None,
            'power_coverage_seconds':self.power_seconds,
            'notes':['GPU-hours = assigned GPU count x process wall time; not kernel execution time.',
                'Board energy is trapezoidal estimate from nvidia-smi, includes board idle/other processes.',
                'Host utilization is shared; do not sum host totals across worker attempts.',
                'Running records after hard kill stop at last sample; no exact crash timestamp.']},self.base.with_suffix('.json'))
    def loop(self):
        while not self.stop_event.wait(self.cfg.get('resource_sample_seconds',10)):
            try:self.sample()
            except Exception as e:print('[RESOURCE WARNING]',e,flush=True)
    def stop(self,status='complete',error=None):
        self.stop_event.set();self.thread.join(timeout=5)
        self.sample();self.write_summary(status,error)


Writing /kaggle/working/utkface_A12_code_v1/resources.py


In [9]:
%%writefile /kaggle/working/utkface_A12_code_v1/bundle.py

"""Một JSON tự chứa toàn bộ kết quả phân tích; chỉ scheduler ghi, không có writer tranh chấp."""
import base64
import hashlib
import json
import math
import os
import time
from pathlib import Path
import numpy as np
import pandas as pd

from specs import experiment_specs, build_plan

def clean(value):
    if isinstance(value, dict):
        return {str(k): clean(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [clean(v) for v in value]
    if isinstance(value, np.ndarray):
        return clean(value.tolist())
    if isinstance(value, np.generic):
        return clean(value.item())
    if isinstance(value, float) and not math.isfinite(value):
        return None
    if hasattr(value,"isoformat"): return value.isoformat()
    return value


def atomic_npz(path, **arrays):
    path = Path(path)
    tmp = path.with_name(path.name + '.tmp')
    with tmp.open('wb') as f:
        np.savez_compressed(f, **arrays)
    os.replace(tmp, path)


def read_file(path):
    ext = path.suffix.lower()
    if ext == '.json':
        return json.loads(path.read_text(encoding='utf-8'))
    if ext == '.csv':
        try: frame = pd.read_csv(path)
        except pd.errors.EmptyDataError: frame = pd.DataFrame()
        return {'columns': frame.columns.tolist(), 'row_count': len(frame), 'rows': clean(frame.to_dict('records'))}
    if ext == '.npz':
        with np.load(path, allow_pickle=False) as z:
            return {'format': 'numpy_arrays', 'arrays': {
                k: {'dtype': str(z[k].dtype), 'shape': list(z[k].shape), 'values': clean(z[k])}
                for k in z.files}}
    if ext == '.jsonl':
        rows=[]; incomplete=0
        lines=path.read_text(encoding='utf-8').splitlines()
        for i,line in enumerate(lines):
            if not line.strip():continue
            try:rows.append(json.loads(line))
            except json.JSONDecodeError:
                if i==len(lines)-1:incomplete+=1
                else:raise
        return {'format':'jsonlines','rows':rows,'incomplete_trailing_records':incomplete}
    if ext == '.xlsx':
        import openpyxl
        formulas=openpyxl.load_workbook(path,read_only=True,data_only=False)
        cached=openpyxl.load_workbook(path,read_only=True,data_only=True)
        try:
            return {'format':'xlsx','sheets':{
                name:{'rows':[[clean(v) for v in row] for row in formulas[name].iter_rows(values_only=True)],
                      'cached_values':[[clean(v) for v in row] for row in cached[name].iter_rows(values_only=True)]}
                for name in formulas.sheetnames}}
        finally:formulas.close();cached.close()
    if ext == '.npy':
        a=np.load(path,allow_pickle=False)
        return {'dtype':str(a.dtype),'shape':list(a.shape),'values':clean(a)}
    if ext in ('.py', '.txt', '.log', '.md'):
        return path.read_text(encoding='utf-8', errors='replace')
    if ext in ('.png', '.jpg', '.jpeg', '.svg'):
        raw = path.read_bytes()
        return {'encoding': 'base64', 'size_bytes': len(raw),
                'sha256': hashlib.sha256(raw).hexdigest(),
                'content': base64.b64encode(raw).decode('ascii')}
    if ext == '.pt':
        return {'storage':'external_checkpoint','relative_path':str(path),
            'size_bytes':path.stat().st_size,'note':'Model/optimizer tensors are kept separately for resume; not duplicated as JSON.'}
    raw=path.read_bytes()
    return {'encoding':'base64','size_bytes':len(raw),'sha256':hashlib.sha256(raw).hexdigest(),
            'content':base64.b64encode(raw).decode('ascii')}


def export_all_results(cfg, stage='running', error=None):
    """One atomic streamed JSON: all output files except checkpoint tensor bytes/self/temp.
    Partial snapshots include all not-started runs. No full-study arrays held in RAM.
    """
    root=Path(cfg['out']);root.mkdir(parents=True,exist_ok=True)
    plan=build_plan(cfg);specs=plan['specs'];seeds=plan['training_seeds'];expected=plan['total_train_runs']
    target=root/'all_results.json';tmp=root/'all_results.json.tmp'
    warnings=[];records=[];n_train=n_test=0
    def safe(path):
        try:return read_file(path)
        except Exception as exc:
            warnings.append({'path':str(path.relative_to(root)),'error':f'{type(exc).__name__}: {exc}'})
            return {'read_pending_or_error':True}
    def eligible(path):
        return path.is_file() and path.name!='all_results.json' and path.suffix!='.tmp' and '__pycache__' not in path.parts
    def dump(f,obj):json.dump(clean(obj),f,ensure_ascii=False,allow_nan=False,separators=(',',':'))
    def tree(f,directory):
        f.write('{');first=True
        for path in sorted(directory.rglob('*')) if directory.exists() else []:
            if not eligible(path):continue
            if not first:f.write(',')
            first=False;dump(f,str(path.relative_to(directory)));f.write(':');dump(f,safe(path))
        f.write('}')
    start=time.perf_counter()
    with tmp.open('w',encoding='utf-8') as f:
        f.write('{"schema_version":"utkface_A12_multiseed_v1","generated_at_utc":')
        dump(f,time.strftime('%Y-%m-%dT%H:%M:%SZ',time.gmtime()))
        f.write(',"stage":');dump(f,stage);f.write(',"error":');dump(f,error)
        f.write(',"configuration":');dump(f,{k:v for k,v in cfg.items() if not k.startswith('_')})
        f.write(',"scope":');dump(f,{
            'training_seeds':seeds,'expected_train_runs':expected,'expected_test_runs':expected,
            'ablation_formulas_and_questions':specs,
            'embedded':'Every output JSON/CSV/XLSX sheet/NPZ/JSONL/log/source/figure, splits, data audit, histories, resource attempts and summary.',
            'excluded':'Only checkpoint/model/optimizer .pt tensor bytes and external dataset/image cache. Checkpoint locations and sizes included.',
            'resume':'Keep full output folder including best.pt/last.pt and original UTKFace input. JSON alone cannot resume.',
            'nonfinite_policy':'Undefined values are null, never NaN/Infinity.',
            'diagnostics':'Actual classification and ordinal-term gradients wrt logits, including O*grad(lambda) for A9. Sampled batches only; not CNN parameter gradients.',
            'history':'Online training accuracy uses augmentation/dropout. Full validation confusion/per-class metrics every epoch.',
            'resources':'Per-attempt process time/CPU/RAM/GPU samples include failures and replay; hard kills bounded by last sample. No invented monetary rate or FLOPs.',
            'statistics':'Three training seeds on one fixed image split, not three independent datasets. Pixel-disjoint, not confirmed person-disjoint.'})
        f.write(',"study_files":{');first=True
        for path in sorted(root.iterdir()):
            if not eligible(path):continue
            if not first:f.write(',')
            first=False;dump(f,path.name);f.write(':');dump(f,safe(path))
        f.write('},"methods":{')
        for i,spec in enumerate(specs):
            if i:f.write(',')
            dump(f,spec['id']);f.write(':{"spec":');dump(f,spec);f.write(',"runs":{')
            for si,seed in enumerate(seeds):
                if si:f.write(',')
                runid=f"{spec['id']}_s{seed}";r=root/'runs'/runid
                trained=(r/'done.json').exists();tested=(r/'test_metrics.json').exists()
                n_train+=trained;n_test+=tested
                status='test_complete' if tested else 'train_complete' if trained else 'started_incomplete' if r.exists() else 'not_started'
                dump(f,str(seed));f.write(':{"run_id":');dump(f,runid)
                f.write(',"status":');dump(f,status);f.write(',"files":');tree(f,r);f.write('}')
                row=dict(method=spec['id'],seed=seed,run_id=runid,status=status)
                for split in ['val','test']:
                    if (r/f'{split}_metrics.json').exists():
                        m=safe(r/f'{split}_metrics.json')
                        row[split]={k:m.get(k) for k in ['qwk','accuracy','balanced_accuracy','f1_macro','group_mae','ece15','nll']}
                records.append(row)
            f.write('}}')
        f.write('}')
        # Every additional top-level directory is included, including future outputs.
        for folder in sorted(root.iterdir()):
            if folder.is_dir() and folder.name not in ['runs','__pycache__']:
                f.write(',');dump(f,folder.name);f.write(':');tree(f,folder)
        f.write(',"progress":');dump(f,dict(train_complete=n_train,test_complete=n_test,planned=expected,
            experiment_complete=stage=='complete' and n_train==expected and n_test==expected and not warnings,runs=records))
        f.write(',"snapshot_read_warnings":');dump(f,warnings)
        f.write(',"export_seconds":');dump(f,time.perf_counter()-start)
        f.write('}\n');f.flush();os.fsync(f.fileno())
    os.replace(tmp,target)
    print(f'[JSON] train={n_train}/{expected} test={n_test}/{expected} stage={stage} size={target.stat().st_size/1024**2:.1f} MiB warnings={len(warnings)}',flush=True)
    if stage=='complete' and warnings:raise RuntimeError('JSON complete có lỗi đọc file; xem snapshot_read_warnings.')
    return target


Writing /kaggle/working/utkface_A12_code_v1/bundle.py


In [10]:
%%writefile /kaggle/working/utkface_A12_code_v1/worker.py

"""Một tiến trình = một GPU = một lượt train; có checkpoint giữa epoch."""

import os

os.environ.setdefault("OMP_NUM_THREADS", "2")
os.environ.setdefault("MKL_NUM_THREADS", "2")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
import sys, json, time, random, traceback, gc, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from data import atomic_json, fingerprint, epoch_batches
from models import OrdinalModel, preprocess
from losses import LossBank, predict, sampled_logit_gradients
from metrics import compute_metrics, checkpoint_key
from bundle import atomic_npz


def seed_all(seed, deterministic=True):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.set_num_threads(2)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = deterministic
    torch.use_deterministic_algorithms(deterministic, warn_only=True)


def rng_state():
    return dict(
        python=random.getstate(),
        numpy=np.random.get_state(),
        torch=torch.get_rng_state(),
        cuda=torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
    )


def restore_rng(state):
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch"])
    if state["cuda"] is not None and torch.cuda.is_available():
        torch.cuda.set_rng_state_all(state["cuda"])


def save_torch(obj, path):
    path = Path(path)
    tmp = path.with_name(path.name + ".tmp")
    if shutil.disk_usage(path.parent).free < 500 * 1024**2:
        raise OSError("Dung lượng trống <500 MiB; dừng trước khi ghi hỏng checkpoint.")
    with tmp.open("wb") as f:
        torch.save(obj, f)
        f.flush(); os.fsync(f.fileno())
    os.replace(tmp, path)
    if path.name=='last.pt':
        atomic_json({k:obj[k] for k in ['epoch','cursor','best_key','bad_epochs','elapsed','peak_gib','fingerprint','spec','seed']},
                    path.parent/'checkpoint_progress.json')


def sync(device):
    if device.type == "cuda":
        torch.cuda.synchronize(device)


def amp_context(device):
    return torch.autocast(
        device_type=device.type, dtype=torch.float16, enabled=device.type == "cuda"
    )


def make_scaler(enabled):
    try:
        return torch.amp.GradScaler("cuda", enabled=enabled, init_scale=1024.0)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=enabled, init_scale=1024.0)


@torch.inference_mode()
def evaluate(model, frame, images, cfg, device, batch_size):
    model.eval()
    preds = []
    probs = []
    ys = []
    ids = []
    cross = 0
    prob_valid = True
    sync(device)
    t0 = time.perf_counter()
    for start in range(0, len(frame), batch_size):
        part = frame.iloc[start : start + batch_size]
        idx = part.cache_index.to_numpy()
        x = preprocess(np.array(images[idx], copy=True), device, False, 0)
        with amp_context(device):
            out = model(x)
        pred, p, c = predict(out, model.family)
        preds.append(pred.cpu().numpy())
        ys.append(part.label.to_numpy())
        ids.append(idx)
        cross += c * len(part)
        if p is None:
            prob_valid = False
        else:
            probs.append(p.cpu().numpy())
    sync(device)
    elapsed = time.perf_counter() - t0
    y = np.concatenate(ys)
    prediction = np.concatenate(preds)
    prob = np.concatenate(probs) if prob_valid else None
    m = compute_metrics(y, prediction, prob)
    m.update(
        threshold_crossing_rate=cross / len(frame),
        evaluation_seconds=elapsed,
        seconds_per_image_including_io=elapsed / len(frame),
        probability_metrics_available=bool(prob_valid),
    )
    arrays = dict(y=y, pred=prediction, cache_index=np.concatenate(ids))
    if prob is not None:
        arrays["probabilities"] = prob
    return m, arrays


def load_resume(path):
    # Chỉ dùng checkpoint do notebook này tạo. weights_only=False để khôi phục RNG.
    return torch.load(path, map_location="cpu", weights_only=False)


def run_impl(jobfile):
    job = json.loads(Path(jobfile).read_text())
    cfg = job["cfg"]
    spec = job["spec"]
    run_dir = Path(job["run_dir"])
    run_dir.mkdir(parents=True, exist_ok=True)
    fingerprint_id = job["fingerprint"]
    seed = job["seed"]
    mode = job.get("mode", "train")
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    if device.type == "cpu" and not cfg.get("smoke_testing", False):
        raise RuntimeError("Cần bật GPU trên Kaggle trước khi train.")
    seed_all(seed, cfg["deterministic"])
    print(f"WORKER START | mode={mode} | seed={seed} | device={device}", flush=True)
    micro = job.get("microbatch", cfg["batch_size"])
    atomic_json(
        {"microbatch": micro, "device": str(device), "torch": torch.__version__, "seed": seed},
        run_dir / "runtime.json",
    )
    donefile = run_dir / "done.json"
    if mode == "train" and donefile.exists():
        if json.loads(donefile.read_text())["fingerprint"] != fingerprint_id:
            raise ValueError("Fingerprint của kết quả cũ không khớp.")
        return
    manifest = pd.read_csv(Path(cfg["out"]) / "data/manifest.csv")
    train = manifest[manifest.split == "train"]
    val = manifest[manifest.split == "val"]
    split = json.loads((Path(cfg["out"]) / "data/split_report.json").read_text())
    images = np.load(split["cache_path"], mmap_mode="r")
    labels = manifest.set_index("cache_index").label.to_dict()
    counts = train.label.value_counts().reindex(range(8), fill_value=0).to_numpy()
    model = OrdinalModel(cfg, spec["family"]).to(device)
    params = sum(p.numel() for p in model.parameters())
    criterion = LossBank(spec, counts, cfg["beta"], cfg["gamma"]).to(device)
    if mode == "test":
        print("TEST START | n=" + str(len(manifest[manifest.split == "test"])), flush=True)
        lock = Path(cfg["out"]) / "selection_lock.json"
        if not lock.exists():
            raise RuntimeError("Chưa khóa lựa chọn; không được đọc test.")
        locked = json.loads(lock.read_text())
        if job["run_id"] not in locked["test_run_ids"]:
            raise RuntimeError("Run không nằm trong danh sách đánh giá đã khóa.")
        state = load_resume(run_dir / "best.pt")
        if state["fingerprint"] != fingerprint_id:
            raise ValueError("Checkpoint không khớp cấu hình.")
        model.load_state_dict(state["model"])
        test = manifest[manifest.split == "test"]
        metrics, arrays = evaluate(
            model, test, images, cfg, device, min(micro, cfg["eval_batch_size"])
        )
        # Đo forward-only sau warmup, không tính decode/CPU và truyền dữ liệu.
        x = preprocess(
            np.array(images[test.cache_index.to_numpy()[: min(32, micro, len(test))]], copy=True),
            device,
            False,
            0,
        )
        with torch.inference_mode(), amp_context(device):
            for _ in range(3):
                model(x)
            sync(device)
            t0 = time.perf_counter()
            for _ in range(cfg.get("timing_repeats", 20)):
                model(x)
            sync(device)
            forward_time = time.perf_counter() - t0
        metrics["forward_timing_batch_size"] = len(x)
        metrics["forward_timing_repeats"] = cfg.get("timing_repeats",20)
        metrics["forward_ms_per_image"] = (
            1000 * forward_time / (len(x) * cfg.get("timing_repeats", 20))
        )
        metrics["forward_images_per_second"] = 1000 / metrics["forward_ms_per_image"]
        metrics["peak_gpu_allocated_gib"] = torch.cuda.max_memory_allocated(device)/1024**3 if device.type=="cuda" else 0.0
        metrics.update(
            run_id=job["run_id"],
            seed=seed,
            method=spec["id"],
            fingerprint=fingerprint_id,
            parameters=params,
        )
        atomic_npz(run_dir / "test_predictions.npz", **arrays)
        atomic_json(metrics, run_dir / "test_metrics.json")
        print(f"TEST DONE | QWK={metrics['qwk']:.5f} | Acc={metrics['accuracy']:.4f} | MAE={metrics['group_mae']:.4f} | F1={metrics['f1_macro']:.4f}", flush=True)
        return
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=cfg["learning_rate"], weight_decay=cfg["weight_decay"]
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.5, patience=3, min_lr=1e-6
    )
    scaler = make_scaler(device.type == "cuda")
    epoch0 = 0
    cursor = 0
    best_key = (-2.0, -1e9, -1.0)
    bad_epochs = 0
    history = []
    full_history = []
    partial = {"sum_loss": 0.0, "n": 0, "grad_sum": 0.0, "steps": 0, "correct": 0, "sum_C": 0.0, "sum_O": 0.0, "sum_lambda": 0.0, "sum_weighted_O": 0.0}
    samples = []
    elapsed_prior = 0.0
    peak_prior = 0.0
    last = run_dir / "last.pt"
    if last.exists():
        state = load_resume(last)
        if state["fingerprint"] != fingerprint_id:
            raise ValueError("Không thể resume: dataset/config/source khác.")
        model.load_state_dict(state["model"])
        optimizer.load_state_dict(state["optimizer"])
        scheduler.load_state_dict(state["scheduler"])
        scaler.load_state_dict(state["scaler"])
        epoch0 = state["epoch"]
        cursor = state["cursor"]
        best_key = tuple(state["best_key"])
        bad_epochs = state["bad_epochs"]
        history = state["history"]
        full_history = state.get("full_history", [])
        partial = state["partial"]
        samples = state["samples"]
        elapsed_prior = state["elapsed"]
        peak_prior = state.get("peak_gib", 0.0)
        restore_rng(state["rng"])
        print(f"RESUME | completed_epochs={epoch0} | next_batch={cursor+1} | best_QWK={best_key[0]:.5f}", flush=True)
        del state
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)
    sync(device)
    start_clock = time.perf_counter()

    def checkpoint_state(epoch, cursor):
        sync(device)
        peak = torch.cuda.max_memory_allocated(device) / 1024**3 if device.type == "cuda" else 0.0
        return {
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
            "scaler": scaler.state_dict(),
            "rng": rng_state(),
            "epoch": epoch,
            "cursor": cursor,
            "best_key": best_key,
            "bad_epochs": bad_epochs,
            "history": history,
            "full_history": full_history,
            "partial": partial,
            "samples": samples,
            "elapsed": elapsed_prior + time.perf_counter() - start_clock,
            "peak_gib": max(peak, peak_prior),
            "fingerprint": fingerprint_id,
            "spec": spec,
            "seed": seed,
        }

    if not last.exists():
        save_torch(checkpoint_state(0, 0), last)
    last_checkpoint_clock = time.monotonic()
    # Nếu chết ngay sau early stopping nhưng trước done.json, không train thêm epoch.
    for epoch in range(epoch0, cfg["epochs"]):
        if bad_epochs >= cfg["patience"]:
            break
        model.train()
        batches = epoch_batches(
            train, cfg["batch_size"], seed + 10007 * epoch, balanced=spec.get("balanced", False)
        )
        tick = max(1, int(np.ceil(len(batches) / 10)))
        epoch_clock = time.perf_counter()
        lr_used = optimizer.param_groups[0]["lr"]
        print(f"EPOCH {epoch+1:02d}/{cfg['epochs']} START | batches={len(batches)} | lr={lr_used:.2e} | microbatch={micro}", flush=True)
        for step in range(cursor, len(batches)):
            idx = batches[step]
            y = torch.tensor([labels[int(i)] for i in idx], device=device, dtype=torch.long)
            x = preprocess(
                np.array(images[idx], copy=True), device, True, seed + epoch * 100000 + step
            )
            before_batch_rng = rng_state()
            # Cho phép AMP scaler giảm và retry cùng batch nếu gradient overflow.
            max_amp_attempts = 20
            for amp_attempt in range(max_amp_attempts):
                optimizer.zero_grad(set_to_none=True)
                with amp_context(device):
                    out = model(x, microbatch=micro)
                gen = torch.Generator(device=device).manual_seed(
                    seed + epoch * 100000 + step + 90000000
                )
                loss, diag = criterion(out, y, generator=gen)
                if not torch.isfinite(loss):
                    raise FloatingPointError(f"Loss không hữu hạn ở epoch {epoch+1}, batch {step}.")
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                grad = torch.nn.utils.clip_grad_norm_(model.parameters(), cfg["gradient_clip"])
                if torch.isfinite(grad):
                    scaler.step(optimizer)
                    scaler.update()
                    break
                if device.type != "cuda" or amp_attempt == max_amp_attempts - 1:
                    raise FloatingPointError(
                        f"Gradient không hữu hạn sau retry ở epoch {epoch+1}, batch {step}."
                    )
                # AMP overflow: giảm scale và chạy lại CÙNG batch, không bỏ qua dữ liệu.
                scaler.update(new_scale=max(scaler.get_scale() / 2, 1e-6))
                restore_rng(before_batch_rng)
                del loss, out, diag
            del before_batch_rng
            partial["sum_loss"] += float(loss.detach()) * len(idx)
            partial["n"] += len(idx)
            partial["grad_sum"] += float(grad)
            partial["steps"] += 1
            partial["correct"] += int((out["logits"].detach().argmax(1) == y).sum())
            for key, name in [("C", "sum_C"), ("O", "sum_O"), ("lam", "sum_lambda"), ("weighted_O", "sum_weighted_O")]:
                partial[name] += float(diag[key].sum())

            if (step + 1) % tick == 0 or step == len(batches) - 1:
                if "lam" in diag:
                    keys = ["C", "O", "neff", "s", "error_sum", "lam", "weighted_O"]
                    grads = sampled_logit_gradients(
                        out["logits"], y, criterion
                    )
                    data = (
                        torch.cat([torch.stack([diag[k] for k in keys], 1), grads], 1).cpu().numpy()
                    )
                    samples.append(
                        np.column_stack(
                            [
                                np.full(len(idx), epoch + 1),
                                np.full(len(idx), step),
                                np.array([labels[int(i)] for i in idx]),
                                data,
                            ]
                        )
                    )
                epoch_elapsed = time.perf_counter() - epoch_clock
                processed_batches = step - cursor + 1
                eta_seconds = epoch_elapsed / processed_batches * (len(batches) - step - 1)
                print(f"EPOCH {epoch+1:02d}/{cfg['epochs']} | {100*(step+1)/len(batches):5.1f}% | batch={step+1}/{len(batches)} | loss={partial['sum_loss']/partial['n']:.5f} | train_acc={partial['correct']/partial['n']:.4f} | lr={lr_used:.2e} | ETA_epoch={eta_seconds:.0f}s", flush=True)
                atomic_json(
                    {
                        "epoch": epoch + 1,
                        "batch": step + 1,
                        "batches": len(batches),
                        "loss": partial["sum_loss"] / partial["n"],
                        "microbatch": micro,
                        "elapsed_seconds": elapsed_prior + time.perf_counter() - start_clock,
                    },
                    run_dir / "progress.json",
                )
            del loss, out, x, y, diag
            if ((step + 1) % cfg["checkpoint_every_batches"] == 0 or
                    time.monotonic()-last_checkpoint_clock >= cfg.get("checkpoint_seconds",300)):
                save_torch(checkpoint_state(epoch, step + 1), last)
                last_checkpoint_clock=time.monotonic()
            if (Path(cfg["out"]) / "STOP").exists():
                save_torch(checkpoint_state(epoch, step + 1), last)
                return 75
        cursor = 0
        print(f"EPOCH {epoch+1:02d} VALIDATION START", flush=True)
        val_m, val_arrays = evaluate(model, val, images, cfg, device, min(micro, cfg["eval_batch_size"]))
        key = checkpoint_key(val_m)
        improved = key > best_key
        if improved:
            best_key = key
            bad_epochs = 0
            save_torch(
                {
                    "model": model.state_dict(),
                    "fingerprint": fingerprint_id,
                    "epoch": epoch + 1,
                    "validation": val_m,
                    "spec": spec,
                    "seed": seed,
                },
                run_dir / "best.pt",
            )
            atomic_npz(run_dir / "val_predictions.npz", **val_arrays)
            atomic_json(val_m, run_dir / "val_metrics.json")
        else:
            bad_epochs += 1
        scheduler.step(val_m["qwk"] if val_m["qwk"] is not None else -2.0)
        row = {
            "epoch": epoch + 1,
            "train_loss": partial["sum_loss"] / partial["n"],
            "train_accuracy_online": partial["correct"] / partial["n"],
            "train_C_mean": partial["sum_C"] / partial["n"],
            "train_O_mean": partial["sum_O"] / partial["n"],
            "train_lambda_mean": partial["sum_lambda"] / partial["n"],
            "train_weighted_O_mean": partial["sum_weighted_O"] / partial["n"],
            "grad_norm_before_clip": partial["grad_sum"] / partial["steps"],
            "val_qwk": val_m["qwk"],
            "val_accuracy": val_m["accuracy"],
            "val_nll": val_m["nll"],
            "val_ece15": val_m["ece15"],
            "val_group_mae": val_m["group_mae"],
            "val_f1_macro": val_m["f1_macro"],
            "val_balanced_accuracy": val_m["balanced_accuracy"],
            "learning_rate": lr_used,
            "learning_rate_next": optimizer.param_groups[0]["lr"],
            "epoch_seconds_this_session": time.perf_counter() - epoch_clock,
            "validation_seconds": val_m["evaluation_seconds"],
            "train_images_seen": partial["n"],
            "train_steps": partial["steps"],
            "microbatch": micro,
            "best": improved,
        }
        history.append(row)
        tmp_history = run_dir / "history.csv.tmp"
        pd.DataFrame(history).to_csv(tmp_history, index=False)
        os.replace(tmp_history, run_dir / "history.csv")
        full_history.append({**row, "validation_metrics": val_m, "best_qwk_so_far": best_key[0], "early_stopping_bad_epochs": bad_epochs})
        atomic_json(full_history, run_dir / "history_full.json")
        if samples:
            atomic_npz(
                run_dir / f"diagnostics_epoch_{epoch+1:03d}.npz",
                values=np.concatenate(samples),
                columns=np.array(
                    [
                        "epoch",
                        "batch",
                        "class",
                        "C",
                        "O",
                        "neff",
                        "s",
                        "error_sum",
                        "lambda",
                        "weighted_O",
                        "grad_classification_l1",
                        "grad_ordinal_term_l1",
                        "grad_ratio",
                        "grad_cosine",
                    ]
                ),
            )
        samples = []
        partial = {"sum_loss": 0.0, "n": 0, "grad_sum": 0.0, "steps": 0, "correct": 0, "sum_C": 0.0, "sum_O": 0.0, "sum_lambda": 0.0, "sum_weighted_O": 0.0}
        save_torch(checkpoint_state(epoch + 1, 0), last)
        print(f"EPOCH {epoch+1:02d} END | loss={row['train_loss']:.5f} | val_QWK={val_m['qwk']:.5f} | val_Acc={val_m['accuracy']:.4f} | val_MAE={val_m['group_mae']:.4f} | val_F1={val_m['f1_macro']:.4f} | BEST={best_key[0]:.5f} | improved={improved} | patience={bad_epochs}/{cfg['patience']} | seconds={row['epoch_seconds_this_session']:.1f} | checkpoint=saved", flush=True)
        if cfg.get("debug_pause_after_epoch") == epoch + 1:
            return 75
    state = load_resume(run_dir / "best.pt")
    model.load_state_dict(state["model"])
    val_m, arrays = evaluate(model, val, images, cfg, device, min(micro, cfg["eval_batch_size"]))
    atomic_npz(run_dir / "val_predictions.npz", **arrays)
    atomic_json(val_m, run_dir / "val_metrics.json")
    sync(device)
    peak = torch.cuda.max_memory_allocated(device) / 1024**3 if device.type == "cuda" else 0.0
    if cfg.get('evaluate_train_at_best',True):
        print('TRAIN BEST CHECKPOINT EVALUATION (no augmentation/dropout)',flush=True)
        train_m, train_arrays = evaluate(model,train,images,cfg,device,min(micro,cfg['eval_batch_size']))
        atomic_json(train_m,run_dir/'train_metrics.json')
        atomic_npz(run_dir/'train_predictions.npz',**train_arrays)
    done = {
        "run_id": job["run_id"],
        "method": spec["id"],
        "family": spec["family"],
        "seed": seed,
        "fingerprint": fingerprint_id,
        "epochs_completed": len(history),
        "best_epoch": state["epoch"],
        "best_validation": val_m,
        "parameters": params,
        "trainable_parameters": sum(p.numel() for p in model.parameters() if p.requires_grad),
        "optimizer": "AdamW",
        "class_counts": counts.tolist(),
        "class_weights": criterion.weights.detach().cpu().tolist(),
        "training_and_validation_seconds": elapsed_prior + time.perf_counter() - start_clock,
        "gpu_count": int(device.type == "cuda"),
        "peak_gpu_allocated_gib": max(peak, peak_prior),
        "microbatch": micro,
        "logical_batch": cfg["batch_size"],
        "gpu_name": torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU_SMOKE_ONLY",
    }
    atomic_json(done, donefile)
    print(f"TRAIN DONE | best_epoch={done['best_epoch']} | val_QWK={val_m['qwk']:.5f} | epochs={len(history)} | minutes={done['training_and_validation_seconds']/60:.1f}", flush=True)
    if not cfg.get("keep_completed_optimizer", False):
        last.unlink(missing_ok=True)
    return 0


def run(jobfile):
    from resources import ResourceMonitor
    job=json.loads(Path(jobfile).read_text())
    monitor=ResourceMonitor(Path(job['run_dir'])/'resources',job['cfg'],job.get('mode','train'),job)
    try:
        code=run_impl(jobfile)
        monitor.stop('paused' if code==75 else 'complete')
        return code
    except BaseException as exc:
        monitor.stop('failed',f'{type(exc).__name__}: {exc}')
        raise

if __name__ == "__main__":
    try:
        code = run(sys.argv[1])
        sys.exit(code or 0)
    except torch.cuda.OutOfMemoryError:
        traceback.print_exc()
        sys.exit(42)
    except Exception:
        traceback.print_exc()
        sys.exit(1)


Writing /kaggle/working/utkface_A12_code_v1/worker.py


In [11]:
%%writefile /kaggle/working/utkface_A12_code_v1/schedule.py

"""Lập danh sách thực nghiệm, chạy hai GPU độc lập, khóa lựa chọn trước test."""

import os, sys, json, time, shutil, subprocess, hashlib
from pathlib import Path
from collections import deque
from tqdm.auto import tqdm
from data import atomic_json, fingerprint, prepare_data


from specs import experiment_specs, build_plan


def config_identity(cfg, split_report):
    # Đường dẫn/thiết bị và checkpoint frequency không thay đổi bài toán tối ưu.
    exclude = {
        "out",
        "cache_root",
        "data_root",
        "resume_from",
        "weights_path",
        "checkpoint_every_batches",
        "keep_completed_optimizer",
        "eval_batch_size",
        "timing_repeats",
        "gpu_ids",
        "debug_pause_after_epoch",
        "json_snapshot_seconds",
        "log_poll_seconds", "checkpoint_seconds", "max_session_hours", "shutdown_margin_seconds",
        "resource_sample_seconds", "gpu_hour_price_usd", "_deadline", "export_xlsx",
    }
    critical = {k: v for k, v in cfg.items() if k not in exclude and not k.startswith("_")}
    here = Path(__file__).parent
    sources = {
        p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted(here.glob("*.py"))
    }
    return fingerprint(
        {
            "config": critical,
            "split": split_report["split_fingerprint"],
            "sources": sources,
            "weights_sha256": cfg.get("weights_sha256", "SMOKE_ONLY"),
        }
    )


def restore_outputs(cfg):
    source = cfg.get("resume_from", "")
    if not source:
        return
    src = Path(source)
    dst = Path(cfg["out"])
    if not src.is_dir():
        raise FileNotFoundError(
            "resume_from phải là thư mục output cũ đã Add Input, không phải ZIP chưa giải nén."
        )
    if src.resolve() == dst.resolve():
        return
    if not (src / "config.json").exists():
        raise ValueError("Chọn đúng thư mục chứa config.json, runs và data.")
    old_config=json.loads((src/'config.json').read_text())
    if old_config.get('schema_version')!='utkface_A12_multiseed_v1':
        raise ValueError('Output cũ không thuộc bộ A1–A12 ba seed. Không resume checkpoint bộ E cũ.')
    if any(dst.glob("runs/*/done.json")) or any(dst.glob("runs/*/last.pt")):
        print("Đã có kết quả tại out; giữ nguyên, không chép đè từ resume_from.")
        return
    print("Khôi phục output đã lưu từ phiên trước...")
    shutil.copytree(
        src,
        dst,
        dirs_exist_ok=True,
        ignore=shutil.ignore_patterns(
            "*.tmp", "results_full.zip", "results_light.zip", "STOP"
        ),
    )
    # Fingerprint được kiểm tra lại trước khi train; cache được xây lại nếu đường dẫn cũ mất.


def ensure_weights(cfg):
    if cfg.get("smoke_testing"):
        return
    import torch
    from torchvision.models import ResNeXt50_32X4D_Weights

    cached=Path(cfg['out'])/'assets'/'resnext50_32x4d-1a0047aa.pth'
    if not cfg.get('weights_path') and cached.is_file(): cfg['weights_path']=str(cached)
    if cfg.get("weights_path"):
        path = Path(cfg["weights_path"])
        if not path.is_file():
            raise FileNotFoundError("Không tìm thấy pretrained checkpoint được chỉ định.")
    else:
        weights = ResNeXt50_32X4D_Weights.IMAGENET1K_V2
        try:
            weights.get_state_dict(progress=True, check_hash=True)
        except Exception as e:
            raise RuntimeError(
                "Không tải được ImageNet weights. Bật Internet trên Kaggle hoặc gán weights_path tới file resnext50_32x4d-1a0047aa.pth đã Add Input. Không tự chuyển sang random init."
            ) from e
        path = Path(torch.hub.get_dir()) / "checkpoints" / Path(weights.url).name
    cached.parent.mkdir(parents=True,exist_ok=True)
    if path.resolve()!=cached.resolve(): shutil.copy2(path,cached)
    path=cached
    cfg["weights_path"] = str(path)
    h = hashlib.sha256()
    with path.open("rb") as f:
        for b in iter(lambda: f.read(2**20), b""):
            h.update(b)
    cfg["weights_sha256"] = h.hexdigest()


def make_job(cfg, spec, seed, common, mode="train"):
    run_id = f"{spec['id']}_s{seed}"
    folder = Path(cfg["out"]) / "runs" / run_id
    ident = fingerprint({"common": common, "spec": spec, "seed": seed})
    return dict(
        cfg=cfg,
        spec=spec,
        seed=seed,
        run_id=run_id,
        run_dir=str(folder),
        fingerprint=ident,
        mode=mode,
        microbatch=cfg["batch_size"],
    )


def dispatch(jobs, cfg, description):
    """Two independent workers; tail their plain-text logs into Kaggle saved-version stdout."""
    import torch
    from bundle import export_all_results
    gpu_ids = cfg.get('gpu_ids') or list(range(torch.cuda.device_count()))
    if not gpu_ids and cfg.get('smoke_testing'):
        gpu_ids = [None]
    if not gpu_ids:
        raise RuntimeError('Bật GPU T4 x2 trong Kaggle Settings.')
    gpu_ids = gpu_ids[:2]
    queue = deque()
    completed = 0
    for job in jobs:
        folder = Path(job['run_dir'])
        target = folder / ('done.json' if job['mode'] == 'train' else 'test_metrics.json')
        if target.exists():
            if json.loads(target.read_text()).get('fingerprint') != job['fingerprint']:
                raise ValueError(f"Code/cấu hình thay đổi: {job['run_id']}. Dùng OUT mới.")
            if job['mode'] == 'train' and not (folder / 'best.pt').is_file():
                raise FileNotFoundError(f"{job['run_id']} thiếu best.pt; không thể đánh giá test. ZIP nhẹ không dùng để resume.")
            completed += 1
            print(f"[SKIP] {job['run_id']} {job['mode']}: đã hoàn tất.", flush=True)
        else:
            queue.append(job)
    print(f'[{description}] {completed}/{len(jobs)} đã xong | workers={len(gpu_ids)}', flush=True)
    active, failures = {}, []
    paused = False
    script = Path(__file__).with_name('worker.py')
    last_export = time.monotonic()
    last_heartbeat = time.monotonic()
    stage = jobs[0]['mode'] if jobs else 'running'
    def relay(item):
        text = item['reader'].read()
        if text:
            # Prefix worker logs; keep complete original logs on disk as well.
            for line in text.replace('\r', '\n').splitlines():
                if line.strip():
                    print(f"[GPU {item['gpu']} | {item['job']['run_id']}] {line}", flush=True)
    try:
        while queue or active:
            if (Path(cfg['out'])/'STOP').exists() or time.monotonic()>=cfg.get('_deadline',float('inf')):
                paused=True; queue.clear(); (Path(cfg['out'])/'STOP').touch()
            for gpu in gpu_ids:
                if gpu in active or not queue: continue
                job = queue.popleft(); job['physical_gpu_id']=gpu; folder = Path(job['run_dir']); folder.mkdir(parents=True, exist_ok=True)
                runtime = folder / 'runtime.json'
                if runtime.exists():
                    job['microbatch'] = min(job['microbatch'], json.loads(runtime.read_text()).get('microbatch',job['microbatch']))
                path = folder / ('job.json' if job['mode']=='train' else 'test_job.json')
                atomic_json(job,path)
                logpath = folder / f"{job['mode']}.log"
                writer = logpath.open('a',encoding='utf-8',buffering=1)
                reader = logpath.open('r',encoding='utf-8',errors='replace'); reader.seek(0,2)
                env = os.environ.copy()
                env.update(OMP_NUM_THREADS='2',MKL_NUM_THREADS='2',OPENBLAS_NUM_THREADS='1',
                    PYTHONHASHSEED=str(job['seed']),CUBLAS_WORKSPACE_CONFIG=':4096:8',PYTHONUNBUFFERED='1')
                if gpu is not None: env['CUDA_VISIBLE_DEVICES']=str(gpu)
                proc = subprocess.Popen([sys.executable,'-u',str(script),str(path)],stdout=writer,stderr=subprocess.STDOUT,env=env)
                active[gpu]={'process':proc,'job':job,'writer':writer,'reader':reader,'gpu':gpu}
                print(f"[START] GPU={gpu} | {job['run_id']} | {job['mode']} | microbatch={job['microbatch']}",flush=True)
            changed = False
            for gpu,item in list(active.items()):
                relay(item)
                proc,job = item['process'],item['job']; code=proc.poll()
                if code is None: continue
                relay(item); item['writer'].close();item['reader'].close();del active[gpu]
                changed=True
                folder=Path(job['run_dir'])
                target=folder/('done.json' if job['mode']=='train' else 'test_metrics.json')
                if code==0 and target.exists():
                    completed+=1
                    print(f"[DONE {completed}/{len(jobs)}] {job['run_id']} {job['mode']}",flush=True)
                elif code==42 and job['microbatch']>2 and not paused:
                    job['microbatch']=max(2,job['microbatch']//2);queue.appendleft(job)
                    print(f"[OOM RETRY] {job['run_id']} microbatch={job['microbatch']}",flush=True)
                elif code==75:
                    paused=True;queue.clear();(Path(cfg['out'])/'STOP').touch()
                else:
                    tail=(folder/f"{job['mode']}.log").read_text(errors='replace')[-4000:]
                    failures.append({'run':job['run_id'],'mode':job['mode'],'exit_code':code,'error_tail':tail})
                    atomic_json(failures,Path(cfg['out'])/'failures.json')
                    print(f"[FAILED] {job['run_id']} exit={code}; các lượt khác vẫn tiếp tục.",flush=True)
            now=time.monotonic()
            if changed or now-last_export>=cfg.get('json_snapshot_seconds',180):
                export_all_results(cfg,stage=stage);last_export=time.monotonic()
            if now-last_heartbeat>=60:
                print(f'[HEARTBEAT] {description}: done={completed}/{len(jobs)} active={len(active)} queued={len(queue)}',flush=True)
                last_heartbeat=now
            if active:time.sleep(cfg.get('log_poll_seconds',2))
        if failures:raise RuntimeError(f'{len(failures)} lượt lỗi; xem failures.json. Chưa mở test nếu train lỗi.')
        if paused:raise InterruptedError('Đã lưu checkpoint và dừng. Chạy lại cùng notebook để tiếp tục.')
    finally:
        # Clean shutdown on interrupt or export/scheduler failure; never leave orphan workers.
        if active:
            (Path(cfg['out'])/'STOP').touch()
            deadline=time.monotonic()+45
            while any(i['process'].poll() is None for i in active.values()) and time.monotonic()<deadline:
                for item in active.values(): relay(item)
                time.sleep(0.5)
            for item in active.values():
                proc=item['process']
                if proc.poll() is None:
                    proc.terminate()
                    try:proc.wait(timeout=5)
                    except subprocess.TimeoutExpired:proc.kill();proc.wait()
                relay(item);item['writer'].close();item['reader'].close()
        export_all_results(cfg,stage='paused' if paused else stage)


def run_all(cfg):
    from resources import ResourceMonitor
    cfg=dict(cfg)
    out=Path(cfg['out']); out.mkdir(parents=True,exist_ok=True)
    started=time.monotonic()
    # Resume uses full output, not just all_results.json. Existing destination wins on rerun.
    restore_outputs(cfg)
    (out/'STOP').unlink(missing_ok=True)
    cfg['_deadline']=started+max(0,cfg.get('max_session_hours',10)*3600-cfg.get('shutdown_margin_seconds',600))
    monitor=ResourceMonitor(out/'sessions',cfg,'scheduler')
    try:
        plan=_run_all(cfg)
        monitor.stop('complete')
    except InterruptedError:
        monitor.stop('paused')
        from reporting import build_partial_report
        build_partial_report(cfg)
        from bundle import export_all_results
        export_all_results(cfg,stage='paused')
        print('[PAUSED] Checkpoints và JSON đã lưu. Save Version output, Add Input output này vào phiên mới; đặt RESUME_FROM.',flush=True)
        return {**build_plan(cfg),'status':'paused'}
    except BaseException as exc:
        monitor.stop('failed',str(exc))
        from bundle import export_all_results
        export_all_results(cfg,stage='failed',error={'type':type(exc).__name__,'message':str(exc)})
        raise
    from bundle import export_all_results
    export_all_results(cfg,stage='complete')
    return {**plan,'status':'complete'}


def _run_all(cfg):
    out=Path(cfg['out'])
    print('[SETUP] Pretrained weights...',flush=True)
    ensure_weights(cfg)
    print('[DATA] Audit / split / cache...',flush=True)
    report=prepare_data(cfg)
    common=config_identity(cfg,report)
    existing=out/'study_identity.json'
    if existing.exists() and json.loads(existing.read_text())['fingerprint']!=common:
        raise ValueError('Config/source/split thay đổi. Dùng OUT mới; không trộn nghiên cứu.')
    atomic_json({'fingerprint':common},existing)
    public_cfg={k:v for k,v in cfg.items() if not k.startswith('_')}
    atomic_json(public_cfg,out/'config.json')
    from resources import hardware
    atomic_json(hardware(),out/'environment.json')
    scripts=out/'source';scripts.mkdir(exist_ok=True)
    for p in Path(__file__).parent.glob('*.py'):shutil.copy2(p,scripts/p.name)
    atomic_json(cfg['implementation_checks'],out/'implementation_checks.json')
    plan=build_plan(cfg);specs=plan['specs'];seeds=plan['training_seeds']
    atomic_json(plan,out/'experiment_plan.json')
    import pandas as pd
    pd.DataFrame(specs).to_csv(out/'ablation_formulas_questions.csv',index=False)
    from bundle import export_all_results
    export_all_results(cfg,stage='training')
    jobs=[make_job(cfg,s,seed,common) for seed in seeds for s in specs]
    dispatch(jobs,cfg,f'TRAIN {len(jobs)} = 18 x {len(seeds)} seeds')
    # One fixed lambda selected from MEAN validation performance over the three seeds.
    scores=[]
    for spec in specs:
        if spec['family']!='A4':continue
        ms=[json.loads((out/'runs'/f"{spec['id']}_s{seed}"/'val_metrics.json').read_text()) for seed in seeds]
        scores.append(dict(method=spec['id'],lam=spec['lam'],
            qwk=sum(m['qwk'] for m in ms)/len(ms),group_mae=sum(m['group_mae'] for m in ms)/len(ms),
            f1_macro=sum(m['f1_macro'] for m in ms)/len(ms)))
    from metrics import checkpoint_key
    winner=max(scores,key=checkpoint_key)['method']
    testjobs=[make_job(cfg,s,seed,common,'test') for seed in seeds for s in specs]
    lock=dict(study_fingerprint=common,training_seeds=seeds,best_fixed_lambda_by_validation=winner,
        fixed_lambda_validation_scores=scores,selection_rule=plan['fixed_lambda_selection'],
        test_run_ids=[j['run_id'] for j in testjobs],primary_metric='qwk',
        created_at_utc=time.strftime('%Y-%m-%dT%H:%M:%SZ',time.gmtime()))
    lockfile=out/'selection_lock.json'
    if lockfile.exists():
        old=json.loads(lockfile.read_text())
        for k in ['study_fingerprint','training_seeds','best_fixed_lambda_by_validation','test_run_ids']:
            if old.get(k)!=lock[k]:raise ValueError('Selection lock không khớp; không chọn lại theo test.')
    else:atomic_json(lock,lockfile)
    dispatch(testjobs,cfg,'TEST: checkpoint và lambda đã khóa bằng validation')
    # Budget check also reserves a new session for reporting if needed.
    if time.monotonic()>=cfg['_deadline']:raise InterruptedError('Tạm dừng trước reporting.')
    from reporting import build_report
    build_report(cfg)
    return plan


Writing /kaggle/working/utkface_A12_code_v1/schedule.py


In [12]:
%%writefile /kaggle/working/utkface_A12_code_v1/reporting.py

"""Tổng hợp sau khi mọi cấu hình đã khóa. Không chọn phương pháp theo test."""

import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
from data import atomic_json, LABELS

SCALARS = [
    "qwk",
    "group_mae",
    "accuracy",
    "balanced_accuracy",
    "f1_macro",
    "precision_macro",
    "recall_macro",
    "f1_weighted",
    "precision_weighted",
    "recall_weighted",
    "within1",
    "severe_error_rate",
    "specificity_macro",
    "mcc",
    "auc_roc_macro",
    "auc_pr_macro",
    "ece15",
    "nll",
]


def qwk_fast(y, pred):
    K = 8
    cm = np.bincount(y * K + pred, minlength=K * K).reshape(K, K).astype(float)
    cost = (np.arange(K)[:, None] - np.arange(K)[None, :]) ** 2
    expected = np.outer(cm.sum(1), cm.sum(0)) / cm.sum()
    den = (cost * expected).sum()
    return 1 - (cost * cm).sum() / den if den > 0 else np.nan


def paired_interval(A, B, n_boot, seed=2026):
    """Lấy mẫu lại cùng image IDs cho 2 phương pháp, trên một training seed cố định.
    CI có điều kiện trên split/seed đã chạy, không đại diện mọi split hay danh tính.
    """
    aa = []
    bb = []
    truth = None
    for a, b in zip(A, B):
        order = np.argsort(a["cache_index"])
        order_b = np.argsort(b["cache_index"])
        if not np.array_equal(a["cache_index"][order], b["cache_index"][order_b]):
            raise ValueError("Prediction không cùng ảnh.")
        if not np.array_equal(a["y"][order], b["y"][order_b]):
            raise ValueError("Nhãn không khớp.")
        y = a["y"][order]
        if truth is not None and not np.array_equal(truth, y):
            raise ValueError("Các seed không cùng test.")
        truth = y
        aa.append(a["pred"][order])
        bb.append(b["pred"][order_b])
    aa = np.asarray(aa)
    bb = np.asarray(bb)
    rng = np.random.default_rng(seed)
    dq = []
    dm = []
    for _ in range(n_boot):
        idx = rng.integers(0, len(truth), len(truth))
        y = truth[idx]
        delta = [qwk_fast(y, pa[idx]) - qwk_fast(y, pb[idx]) for pa, pb in zip(aa, bb)]
        dq.append(np.mean(delta))
        dm.append(np.abs(aa[:, idx] - y).mean() - np.abs(bb[:, idx] - y).mean())
    return {
        "delta_qwk_ci95": np.nanquantile(dq, [0.025, 0.975]).tolist(),
        "delta_mae_ci95": np.quantile(dm, [0.025, 0.975]).tolist(),
        "bootstrap_replicates": n_boot,
        "scope": "Paired image bootstrap, conditional on fixed test split and observed training seeds (mean metric difference); does not estimate training-seed variability; exploratory intervals, not multiplicity-adjusted.",
    }


def plot_cm(cm, title, path):
    fig, ax = plt.subplots(figsize=(6.5, 5.7))
    im = ax.imshow(cm, cmap="Blues")
    ax.set(
        xticks=range(8),
        yticks=range(8),
        xticklabels=LABELS,
        yticklabels=LABELS,
        xlabel="Predicted group",
        ylabel="True group",
        title=title,
    )
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
    for i in range(8):
        for j in range(8):
            ax.text(
                j,
                i,
                str(cm[i][j]),
                ha="center",
                va="center",
                fontsize=8,
                color="white" if cm[i][j] > np.max(cm) * 0.5 else "black",
            )
    fig.colorbar(im, ax=ax, fraction=0.045)
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)



def resource_rows(root):
    rows=[]
    for f in sorted((root/'runs').glob('*/resources/*.json')):
        m=json.loads(f.read_text())
        rows.append({k:m.get(k) for k in ['attempt_id','run_id','seed','mode','status','wall_seconds',
            'process_cpu_seconds','peak_process_rss_gib','assigned_gpu_hours','estimated_cost_usd',
            'sampled_gpu_energy_kwh','power_coverage_seconds','gpu_utilization_sample_mean','physical_gpu_id']})
    return pd.DataFrame(rows)


def build_partial_report(cfg):
    """Only available results. Never pretend incomplete methods have 3 seeds or zero SD."""
    from specs import experiment_specs,build_plan
    from bundle import clean
    root=Path(cfg['out']);folder=root/'summary';folder.mkdir(exist_ok=True)
    rows=[];history=[];classes=[]
    extras=['group_rmse','linear_weighted_kappa','within2','mean_signed_group_error',
        'underestimation_rate','overestimation_rate','brier_multiclass','evaluation_seconds',
        'forward_ms_per_image','forward_images_per_second','seconds_per_image_including_io']
    for spec in experiment_specs():
        for seed in cfg['training_seeds']:
            runid=f"{spec['id']}_s{seed}";r=root/'runs'/runid
            done=json.loads((r/'done.json').read_text()) if (r/'done.json').exists() else {}
            for split in ['train','val','test']:
                file=r/f'{split}_metrics.json'
                if not file.exists():continue
                m=json.loads(file.read_text())
                row=dict(method=spec['id'],seed=seed,run_id=runid,split=split,train_complete=bool(done),
                    **{k:m.get(k) for k in SCALARS+extras})
                elapsed=done.get('training_and_validation_seconds')
                row.update(best_epoch=done.get('best_epoch'),epochs_completed=done.get('epochs_completed'),
                    training_hours=elapsed/3600 if elapsed is not None else None,
                    checkpoint_accounted_gpu_hours=elapsed*done['gpu_count']/3600 if elapsed is not None else None,
                    parameters=done.get('parameters'),peak_gpu_allocated_gib=done.get('peak_gpu_allocated_gib'))
                rows.append(row)
                classes.extend([dict(method=spec['id'],seed=seed,split=split,**c) for c in m['per_class']])
            if (r/'history_full.json').exists():
                for h in json.loads((r/'history_full.json').read_text()):
                    history.append(dict(method=spec['id'],seed=seed,**{k:v for k,v in h.items() if not isinstance(v,(dict,list))}))
    raw=pd.DataFrame(rows);perclass=pd.DataFrame(classes);hist=pd.DataFrame(history)
    formulas=pd.DataFrame(experiment_specs());attempts=resource_rows(root)
    raw.to_csv(folder/'all_runs.csv',index=False)
    perclass.to_csv(folder/'per_class_all.csv',index=False)
    hist.to_csv(folder/'learning_history_all.csv',index=False)
    formulas.to_csv(folder/'ablation_formulas_questions.csv',index=False)
    attempts.to_csv(folder/'resource_attempts.csv',index=False)
    agg_rows=[]
    if len(raw):
        for (method,split),sub in raw.groupby(['method','split'],sort=False):
            row=dict(method=method,split=split,n_observed_seeds=len(sub),
                all_seeds_complete=len(sub)==len(cfg['training_seeds']) and bool(sub.train_complete.all()))
            for key in SCALARS+extras+['training_hours','checkpoint_accounted_gpu_hours','peak_gpu_allocated_gib']:
                a=pd.to_numeric(sub[key],errors='coerce').dropna()
                row[key+'_mean']=a.mean() if len(a) else None
                row[key+'_std']=a.std(ddof=1) if len(a)>1 else None
            agg_rows.append(row)
    aggregate=pd.DataFrame(agg_rows)
    aggregate.to_csv(folder/'mean_std_by_method.csv',index=False)
    atomic_json(clean(agg_rows),folder/'mean_std_by_method.json')
    costs=[]
    if len(attempts):
        for runid,a in attempts.groupby('run_id'):
            row=dict(run_id=runid,attempts=len(a))
            for mode in ['train','test']:
                b=a[a['mode']==mode]
                for k in ['wall_seconds','process_cpu_seconds','assigned_gpu_hours','estimated_cost_usd','sampled_gpu_energy_kwh']:
                    row[mode+'_'+k]=pd.to_numeric(b[k],errors='coerce').sum(min_count=1) if len(b) else None
                row[mode+'_peak_rss_gib']=b.peak_process_rss_gib.max() if len(b) else None
            costs.append(row)
    costframe=pd.DataFrame(costs);costframe.to_csv(folder/'compute_cost_all_attempts.csv',index=False)
    totals=dict(expected_train_runs=build_plan(cfg)['total_train_runs'],
        completed_training_runs=len(list((root/'runs').glob('*/done.json'))),
        completed_test_runs=len(list((root/'runs').glob('*/test_metrics.json'))),
        total_worker_gpu_hours=attempts.assigned_gpu_hours.sum() if len(attempts) else 0,
        total_worker_cpu_hours=attempts.process_cpu_seconds.sum()/3600 if len(attempts) else 0,
        total_estimated_cost_usd=attempts.estimated_cost_usd.sum(min_count=1) if len(attempts) else None,
        gpu_hour_price_usd=cfg.get('gpu_hour_price_usd'),
        scheduler_session_wall_hours=sum(json.loads(p.read_text())['wall_seconds'] for p in (root/'sessions').glob('*.json'))/3600,
        notes=['Attempt ledger includes failed/replayed work; checkpoint elapsed is separate.',
               'CPU time is process user+system core-seconds; host utilization shared.',
               'Wall elapsed of parallel workers must not be interpreted as notebook wall time.',
               'Energy is sampled board estimate, not exact process attribution.',
               'No monetary price supplied => cost null. FLOPs not measured.'])
    atomic_json(clean(totals),folder/'compute_cost_totals.json')
    if cfg.get('export_xlsx',True):
        tmp=folder/'results.tmp.xlsx'
        with pd.ExcelWriter(tmp,engine='openpyxl') as writer:
            for name,frame in [('formulas_questions',formulas),('all_runs',raw),('mean_std',aggregate),
                ('per_class',perclass),('learning_history',hist),('resource_attempts',attempts),('compute_cost',costframe)]:
                frame.to_excel(writer,sheet_name=name,index=False)
            for r in sorted((root/'runs').glob('*/test_metrics.json')):
                m=json.loads(r.read_text())
                pd.DataFrame(m['confusion_matrix'],index=LABELS,columns=LABELS).to_excel(
                    writer,sheet_name=r.parent.name[:31])
        tmp.replace(folder/'results.xlsx')
    return raw,aggregate


def build_report(cfg):
    from specs import experiment_specs
    from bundle import clean
    root=Path(cfg['out']);folder=root/'summary';figures=folder/'figures';figures.mkdir(parents=True,exist_ok=True)
    lock=json.loads((root/'selection_lock.json').read_text());seeds=cfg['training_seeds']
    raw,aggregate=build_partial_report(cfg)
    test=raw[raw.split=='test']
    if len(test)!=len(experiment_specs())*len(seeds):raise ValueError('Chưa đủ test cho tất cả method x seed.')
    predictions={};manifest=pd.read_csv(root/'data/manifest.csv').set_index('cache_index')
    for runid in lock['test_run_ids']:
        r=root/'runs'/runid;m=json.loads((r/'test_metrics.json').read_text())
        with np.load(r/'test_predictions.npz') as z:p={k:z[k].copy() for k in z.files}
        predictions[(m['method'],m['seed'])]=p
        plot_cm(np.array(m['confusion_matrix']),runid,figures/f'confusion_{runid}.png')
        mask=np.abs(p['y']-p['pred'])>=2
        errors=manifest.loc[p['cache_index'][mask],['file','age','label','sha256']].copy()
        errors['prediction']=p['pred'][mask];errors['distance']=np.abs(p['y'][mask]-p['pred'][mask])
        errors.to_csv(r/'severe_errors.csv',index=False)
    # Sum confusion matrices is repeated evaluation of the SAME images, not 3N independent people.
    for spec in experiment_specs():
        cms=[np.array(json.loads((root/'runs'/f"{spec['id']}_s{s}"/'test_metrics.json').read_text())['confusion_matrix']) for s in seeds]
        atomic_json(dict(seeds=seeds,confusion_matrix_sum=np.sum(cms,axis=0).tolist(),
            confusion_matrix_mean=np.mean(cms,axis=0).tolist(),note='Same test images repeated across training seeds; not independent 3N observations.'),
            folder/f"confusion_across_seeds_{spec['id']}.json")
        plot_cm(np.sum(cms,axis=0),spec['id']+' summed across seeds',figures/f"confusion_sum_{spec['id']}.png")
    sub=aggregate[aggregate.split=='test']
    fig,ax=plt.subplots(figsize=(12,4.5))
    ax.bar(sub.method,sub.qwk_mean,yerr=sub.qwk_std,capsize=3,color=['#136f63' if n=='A12' else '#4b79a1' for n in sub.method])
    ax.set(ylabel='Test QWK mean ± sample SD',title='Three training seeds; same fixed test split');ax.tick_params(axis='x',rotation=45)
    fig.tight_layout();fig.savefig(figures/'qwk_mean_std.png',dpi=150);plt.close(fig)
    for seed in seeds:
        fig,axes=plt.subplots(1,2,figsize=(11,4))
        for method in ['A1','A2','A11','A12']:
            h=pd.read_csv(root/'runs'/f'{method}_s{seed}'/'history.csv')
            axes[0].plot(h.epoch,h.train_loss,label=method);axes[1].plot(h.epoch,h.val_qwk,label=method)
        axes[0].set(xlabel='Epoch',ylabel='Training objective (different loss scales)',yscale='log')
        axes[1].set(xlabel='Epoch',ylabel='Validation QWK')
        for ax in axes:ax.legend();ax.grid(alpha=.2)
        fig.tight_layout();fig.savefig(figures/f'learning_curves_s{seed}.png',dpi=150);plt.close(fig)
    # Paired image bootstrap of the mean delta over observed seeds; no seed resampling claims.
    comparisons=[s['id'] for s in experiment_specs() if s['family']!='A4' and s['id']!='A12']+[lock['best_fixed_lambda_by_validation']]
    boot=[];deltas=[]
    for other in comparisons:
        A=[predictions[('A12',s)] for s in seeds];B=[predictions[(other,s)] for s in seeds]
        dq=[];dm=[]
        for seed,a,b in zip(seeds,A,B):
            order=np.argsort(a['cache_index']);ob=np.argsort(b['cache_index'])
            assert np.array_equal(a['cache_index'][order],b['cache_index'][ob]) and np.array_equal(a['y'][order],b['y'][ob])
            q=qwk_fast(a['y'],a['pred'])-qwk_fast(b['y'],b['pred'])
            mae=float(np.abs(a['y']-a['pred']).mean()-np.abs(b['y']-b['pred']).mean())
            dq.append(q);dm.append(mae);deltas.append(dict(comparison=f'A12 minus {other}',seed=seed,delta_qwk=q,delta_mae=mae))
        ci=paired_interval(A,B,cfg['bootstrap_replicates'])
        boot.append(dict(comparison=f'A12 minus {other}',delta_qwk_mean=np.mean(dq),delta_qwk_std=np.std(dq,ddof=1),
            delta_mae_mean=np.mean(dm),delta_mae_std=np.std(dm,ddof=1),**ci))
    atomic_json(clean(boot),folder/'paired_bootstrap.json');pd.DataFrame(boot).to_csv(folder/'paired_bootstrap.csv',index=False)
    pd.DataFrame(deltas).to_csv(folder/'paired_deltas_by_seed.csv',index=False)
    atomic_json(dict(best_fixed_lambda_by_validation=lock['best_fixed_lambda_by_validation'],notes=[
        'Test evaluated only after all training and validation selection lock.',
        'MAE/RMSE in age-group indices, not years.',
        'Mean and sample SD (ddof=1) across three training seeds; same split.',
        'Bootstrap conditional on observed seeds/images; no multiplicity correction.',
        'A5/A7 change lambda magnitude as well as structure.',
        'Latency measured in concurrent jobs; compare cautiously.',
        'Pixel-disjoint is not person-disjoint; no claim of deployment or best method.']),folder/'interpretation_notes.json')
    print('[REPORT] Mean/SD, all per-seed metrics, confusion matrices, XLSX, costs and bootstrap saved.',flush=True)


Writing /kaggle/working/utkface_A12_code_v1/reporting.py


In [13]:
%%writefile /kaggle/working/utkface_A12_code_v1/checks.py

"""Independent formula and gradient checks; no training data or test peeking."""
import json
import torch
from losses import LossBank,ordinal_parts,sampled_logit_gradients
from specs import experiment_specs

def reference(z,y,spec,w,gamma):
    p=z.softmax(1); py=p[torch.arange(len(y)),y];wy=w[y]
    q=torch.stack([p[:,k+1:].sum(1) for k in range(p.shape[1]-1)],1)
    t=y[:,None]>torch.arange(p.shape[1]-1)
    O=-(t*q.log()+(~t)*torch.log1p(-q)).mean(1)
    e=torch.where(t,1-q,q);neff=e.sum(1).square()/e.square().sum(1)
    s=(neff-1)/(p.shape[1]-2);D=(1+s)/2;f=(1-py)**gamma;C=-wy*f*py.log()
    family=spec['family'];lam=wy*D
    if family=='A1':return C.mean()
    if family=='A2':return O.mean()
    if family=='A3':C=torch.zeros_like(C)
    elif family=='A4':lam=torch.full_like(lam,spec['lam'])
    elif family=='A5':lam=wy*neff
    elif family=='A6':lam=D
    elif family=='A7':lam=wy
    elif family=='A8':C=-wy*py.log()
    elif family=='A10':C=-f*py.log()
    elif family=='A11':lam=wy*f*D
    if family!='A9':lam=lam.detach()
    return (C+lam*O).mean()

def run_checks():
    torch.manual_seed(9841);specs=experiment_specs();y=torch.tensor([0,2,4,7])
    for spec in specs:
        bank=LossBank(spec,[30,60,500,220,90,110,40,25]).double()
        z=torch.randn(4,8,dtype=torch.float64,requires_grad=True)
        got,_=bank(z,y);want=reference(z,y,spec,bank.weights,bank.gamma)
        torch.testing.assert_close(got,want,rtol=1e-9,atol=1e-9)
        g=torch.autograd.grad(got,z)[0];gref=torch.autograd.grad(want,z)[0]
        torch.testing.assert_close(g,gref,rtol=1e-8,atol=1e-9)
        diag=sampled_logit_gradients(z,y,bank)
        finite=diag[:,3][torch.isfinite(diag[:,3])]
        assert ((finite>=-1)&(finite<=1)).all()
        z2=(torch.randn(4,8)*1000).requires_grad_(True)
        loss,_=bank(z2,y);loss.backward()
        assert torch.isfinite(loss) and torch.isfinite(z2.grad).all()
    # Same forward, different backward for A9/A12; full derivative captured in diagnostics.
    z=torch.randn(4,8,dtype=torch.float64,requires_grad=True)
    a=LossBank(dict(family='A9'),[100]*8).double();b=LossBank(dict(family='A12'),[100]*8).double()
    la,_=a(z,y);lb,_=b(z,y)
    torch.testing.assert_close(la,lb)
    assert not torch.allclose(torch.autograd.grad(la,z)[0],torch.autograd.grad(lb,z)[0])
    for family in ['A1','A2','A3']:
        d=sampled_logit_gradients(z,y,LossBank(dict(family=family),[100]*8).double())
        if family=='A1':assert (d[:,1]==0).all()
        else:assert (d[:,0]==0).all() and torch.isnan(d[:,2]).all()
    return dict(passed=True,configurations_checked=len(specs),
        checks=['independent_forward_18','independent_backward_18','extreme_logits_18',
                'actual_component_gradient_diagnostics','stop_gradient_A9_A12','zero_component_undefined_ratio'])

if __name__=='__main__':print(json.dumps(run_checks()))


Writing /kaggle/working/utkface_A12_code_v1/checks.py


In [14]:
import importlib, json
for name in ['data','specs','losses','models','metrics','resources','bundle','worker','schedule','reporting','checks']:
    sys.modules.pop(name,None)
importlib.invalidate_caches()
from specs import experiment_specs, build_plan
from checks import run_checks
from IPython.display import display, FileLink
import pandas as pd
CHECKS=run_checks()
assert CHECKS['passed'] and CHECKS['configurations_checked']==18
CFG['implementation_checks']=CHECKS
PLAN=build_plan(CFG)
assert PLAN['total_train_runs']==54
print('Checks:',CHECKS)
print('Training seeds:',TRAINING_SEEDS,'| train:',PLAN['total_train_runs'],'| test:',PLAN['total_test_evaluations'])
print('C=-w_y(1-p_y)^gamma log(p_y); O=mean BCE over K-1 thresholds; D=(1+s)/2; sg=stop-gradient.')
with pd.option_context('display.max_colwidth',None,'display.max_rows',None):
    display(pd.DataFrame(experiment_specs())[['id','formula','question']])


Checks: {'passed': True, 'configurations_checked': 18, 'checks': ['independent_forward_18', 'independent_backward_18', 'extreme_logits_18', 'actual_component_gradient_diagnostics', 'stop_gradient_A9_A12', 'zero_component_undefined_ratio']}
Training seeds: [42, 2006, 2026] | train: 54 | test: 54
C=-w_y(1-p_y)^gamma log(p_y); O=mean BCE over K-1 thresholds; D=(1+s)/2; sg=stop-gradient.


,id,formula,question
0,A1,C,Chỉ C có kém phương pháp đầy đủ không?
1,A2,O,Phương pháp đầy đủ có tốt hơn ordinal thuần không?
2,A3,sg(w_y D) O,Thành phần C đóng góp gì khi giữ nguyên nhánh ordinal?
3,A4_1,C + 0.03 O,Lambda thích nghi có tốt hơn lambda cố định được chọn bằng validation không?
4,A4_2,C + 0.05 O,Lambda thích nghi có tốt hơn lambda cố định được chọn bằng validation không?
5,A4_3,C + 0.1 O,Lambda thích nghi có tốt hơn lambda cố định được chọn bằng validation không?
6,A4_4,C + 0.3 O,Lambda thích nghi có tốt hơn lambda cố định được chọn bằng validation không?
7,A4_5,C + 0.5 O,Lambda thích nghi có tốt hơn lambda cố định được chọn bằng validation không?
8,A4_6,C + 1 O,Lambda thích nghi có tốt hơn lambda cố định được chọn bằng validation không?
9,A4_7,C + 3 O,Lambda thích nghi có tốt hơn lambda cố định được chọn bằng validation không?


In [15]:
from schedule import run_all
from bundle import export_all_results
from data import atomic_json
import traceback
try:
    RESULT=run_all(CFG)
except BaseException as exc:
    failure=dict(type=type(exc).__name__,message=str(exc),traceback=traceback.format_exc())
    atomic_json(failure,Path(OUT)/'last_session_error.json')
    try: export_all_results(CFG,stage='interrupted' if isinstance(exc,KeyboardInterrupt) else 'failed',error=failure)
    except Exception as e: print('Giữ snapshot JSON trước đó:',e,flush=True)
    raise
else:
    (Path(OUT)/'last_session_error.json').unlink(missing_ok=True)
    print('Session status:',RESULT['status'],flush=True)
    print('Output để Add Input khi chạy tiếp:',OUT,flush=True)


[SETUP] Pretrained weights...
Downloading: "https://download.pytorch.org/models/resnext50_32x4d-1a0047aa.pth" to /root/.cache/torch/hub/checkpoints/resnext50_32x4d-1a0047aa.pth


100%|██████████| 95.8M/95.8M [00:00<00:00, 195MB/s]


[DATA] Audit / split / cache...


Kiểm tra và hash ảnh:   0%|          | 0/23708 [00:00<?, ?it/s]

Cache ảnh uint8:   0%|          | 0/23113 [00:00<?, ?it/s]

[JSON] train=0/54 test=0/54 stage=training size=141.7 MiB warnings=0
[TRAIN 54 = 18 x 3 seeds] 0/54 đã xong | workers=2
[START] GPU=0 | A1_s42 | train | microbatch=32
[START] GPU=1 | A2_s42 | train | microbatch=32
[GPU 0 | A1_s42] WORKER START | mode=train | seed=42 | device=cuda:0
[GPU 1 | A2_s42] WORKER START | mode=train | seed=42 | device=cuda:0
[GPU 0 | A1_s42] EPOCH 01/50 START | batches=578 | lr=3.00e-04 | microbatch=32
[GPU 1 | A2_s42] EPOCH 01/50 START | batches=578 | lr=3.00e-04 | microbatch=32
[GPU 0 | A1_s42] EPOCH 01/50 |  10.0% | batch=58/578 | loss=1.31907 | train_acc=0.3168 | lr=3.00e-04 | ETA_epoch=134s
[GPU 1 | A2_s42] EPOCH 01/50 |  10.0% | batch=58/578 | loss=0.43804 | train_acc=0.3292 | lr=3.00e-04 | ETA_epoch=134s
[GPU 0 | A1_s42] EPOCH 01/50 |  20.1% | batch=116/578 | loss=1.18870 | train_acc=0.3575 | lr=3.00e-04 | ETA_epoch=108s
[GPU 1 | A2_s42] EPOCH 01/50 |  20.1% | batch=116/578 | loss=0.38367 | train_acc=0.3661 | lr=3.00e-04 | ETA_epoch=108s
[GPU 0 | A1_s42]

In [16]:
table=Path(OUT)/'summary'/'mean_std_by_method.csv'
if table.exists():
    try: frame=pd.read_csv(table)
    except pd.errors.EmptyDataError: frame=pd.DataFrame()
    if 'split' in frame:
        display(frame.loc[frame['split']=='test',[c for c in ['method','n_observed_seeds','all_seeds_complete','qwk_mean','qwk_std','accuracy_mean','group_mae_mean','f1_macro_mean'] if c in frame]])
for name in ['all_results.json','summary/results.xlsx','summary/compute_cost_totals.json']:
    path=Path(OUT)/name
    if path.exists():
        print(path,'| MiB:',round(path.stat().st_size/1024**2,2));display(FileLink(str(path)))
print('Resume cần toàn bộ thư mục output và dataset gốc; JSON/XLSX riêng lẻ không chứa trọng số model.')


,method,n_observed_seeds,all_seeds_complete,qwk_mean,qwk_std,accuracy_mean,group_mae_mean,f1_macro_mean


/kaggle/working/utkface_A12_3seeds_v1/all_results.json | MiB: 247.2


/kaggle/working/utkface_A12_3seeds_v1/all_results.json

/kaggle/working/utkface_A12_3seeds_v1/summary/results.xlsx | MiB: 0.2


/kaggle/working/utkface_A12_3seeds_v1/summary/results.xlsx

/kaggle/working/utkface_A12_3seeds_v1/summary/compute_cost_totals.json | MiB: 0.0


/kaggle/working/utkface_A12_3seeds_v1/summary/compute_cost_totals.json

Resume cần toàn bộ thư mục output và dataset gốc; JSON/XLSX riêng lẻ không chứa trọng số model.
